# FasterCellNet-RKD v2 — Q2-journal edition (multi-seed · two datasets · modern KD baselines · KD-guided pruning)

**When does difficulty-aware distillation pay off? Knowledge distillation under structured compression for blood-cell classification.**

Teacher: ConvNeXt-Tiny. Student: FasterCellNet (FasterNet-T0 + FX-traceable Coordinate Attention). Datasets: **BloodMNIST** (8 classes) and **Raabin-WBC** (5 WBC classes, plus TestB as an external acquisition-shift set).

### What changed from v1
| v1 (single run) | v2 (this notebook) |
|---|---|
| 1 seed | **3 seeds** (`cfg.seeds`): mean ± std, per-seed McNemar, pooled paired bootstrap, Holm correction |
| baselines: no KD, logit KD | + **DKD** (CVPR'22), **DIST** (NeurIPS'22), **CTKD** (AAAI'23), **WSLD** (ICLR'21, an existing *sample-adaptive* KD); optional logit standardization (CVPR'24) |
| pruning allocation normalised per *stage* (over-pruned stage 2) | allocation normalised **per removed channel**; uniform allocation runs as an ablation |
| one damaged model, then 6-epoch recovery | **KD-guided progressive pruning**: each method guides its own pruning increments inside the same 10-epoch budget |
| levels 20 / 40 / 60 % | **40 / 60 / 75 %** of prunable channels |
| BloodMNIST only | + **Raabin-WBC** (auto-detected Train/TestA/TestB, local resized cache) and a cross-dataset summary |

All baselines use the hyper-parameters from their papers, with **no tuning**. The proposed method is not tuned either. Selection decisions use val only.

### How to run (per dataset)
1. `DATASET = "bloodmnist"` → Run all. The earlier v1 teacher and seed-42 students are **reused** from Drive.
2. Upload the unzipped Raabin-WBC *Double-labeled Cropped Cells* folder to Drive and set `cfg.raabin_root`. Then set `DATASET = "raabin"` → Run all.
3. The cross-dataset cell (S16b) combines both datasets.

Checkpointing works as in v1. **Stage-level:** each stage is saved to Drive once it finishes and is loaded instead of recomputed. **Epoch-level:** every training run resumes at its last finished epoch, and each progressive-pruning increment is saved. A Colab disconnect costs at most one epoch.

**Compute (rough, from the v1 Colab timings):** BloodMNIST ≈ 18 T4-hours (v1 checkpoints reused) and Raabin-WBC ≈ 14 T4-hours. On an A100, expect roughly 3× faster. RQ2 (5 trained methods × 3 levels × 3 seeds) is most of the cost. `MODE = "lean"` runs DIST/CTKD on the main seed only and uses an 8-epoch recovery budget, saving roughly 20 %. Any number of sessions is fine.

### Drive layout
```
/MyDrive/FasterCellNet_RKD/                 ← BloodMNIST (same root as v1 → reuse)
    ckpt/ runs/ preds/ bench/ robustness/ xai/ export/ device_benchmark/ figs/ tables/ stage_status.json
    raabin/   (same structure, Raabin-WBC)
    cross_dataset/
```

---
# S0 · Setup — environment, Drive mount, checkpoint infrastructure
Run every session.

In [ ]:
import subprocess, sys, os, platform
IN_COLAB = "google.colab" in sys.modules

print("Python:", sys.version.split()[0], "|", platform.platform())
try:
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total",
                                   "--format=csv,noheader"]).decode().strip())
except Exception:
    print("No GPU detected — Runtime > Change runtime type > T4 GPU.")

if IN_COLAB or os.environ.get("FCN_INSTALL", "0") == "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist", "timm", "torch-pruning",
                    "onnx", "onnxruntime", "onnxscript", "statsmodels", "thop"], check=False)
    print("Dependencies installed.")

import json, time, copy, math, random, warnings, hashlib, shutil, io
from dataclasses import dataclass, field, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CPU = torch.device("cpu")
print("torch", torch.__version__, "| device:", DEVICE)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ---------------------------------------------------------------------------
# Google Drive = persistence layer. FCN_ROOT env var overrides (used for local tests).
# ---------------------------------------------------------------------------
if os.environ.get("FCN_ROOT"):
    DRIVE_ROOT = Path(os.environ["FCN_ROOT"])
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/FasterCellNet_RKD")
else:
    DRIVE_ROOT = Path("./FasterCellNet_RKD_local")
    print("Not in Colab — using a local folder; checkpoints won't survive a runtime reset.")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive root:", DRIVE_ROOT.resolve())

_MEMO = {}

def set_data_root(dataset):
    """Each dataset gets its own checkpoint tree. BloodMNIST keeps the ORIGINAL root, so checkpoints
    from the earlier (ICCIT-extension) run — teacher + seed-42 RQ1 students — are reused, not retrained."""
    global DATA_ROOT, CKPT_DIR, RUNS_DIR, PREDS_DIR, BENCH_DIR, ROBUST_DIR, XAI_DIR, EXPORT_DIR, \
        DEVICE_DIR, FIGS_DIR, TABLES_DIR, STATUS_PATH
    DATA_ROOT = DRIVE_ROOT if dataset == "bloodmnist" else DRIVE_ROOT / dataset
    CKPT_DIR, RUNS_DIR, PREDS_DIR = DATA_ROOT / "ckpt", DATA_ROOT / "runs", DATA_ROOT / "preds"
    BENCH_DIR, ROBUST_DIR, XAI_DIR = DATA_ROOT / "bench", DATA_ROOT / "robustness", DATA_ROOT / "xai"
    EXPORT_DIR, DEVICE_DIR = DATA_ROOT / "export", DATA_ROOT / "device_benchmark"
    FIGS_DIR, TABLES_DIR = DATA_ROOT / "figs", DATA_ROOT / "tables"
    for d in (CKPT_DIR, RUNS_DIR, PREDS_DIR, BENCH_DIR, ROBUST_DIR, XAI_DIR, EXPORT_DIR, DEVICE_DIR, FIGS_DIR, TABLES_DIR):
        d.mkdir(parents=True, exist_ok=True)
    STATUS_PATH = DATA_ROOT / "stage_status.json"
    _MEMO.clear()
    print(f"[data root] {dataset} → {DATA_ROOT}")

RUN_START = time.time()
def checkpoint_time(label):
    print(f"[TIME] {label}: {(time.time()-RUN_START)/3600:.2f} h elapsed this session")

# ===========================================================================
# Atomic writes — a disconnect mid-save leaves the previous file intact.
# ===========================================================================
def atomic_torch_save(obj, path):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    try:
        os.replace(tmp, path)
    except OSError:
        shutil.move(str(tmp), str(path))

def atomic_write_text(path, text):
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(text)
    try:
        os.replace(tmp, path)
    except OSError:
        shutil.move(str(tmp), str(path))

def torch_load(path, map_location=None):
    return torch.load(path, map_location=map_location or DEVICE, weights_only=False)

# ===========================================================================
# STAGE-LEVEL CHECKPOINTS
# ensure_stage(name, compute, load): memo → Drive status → compute & mark done.
# ===========================================================================

def _load_status():
    try:
        return json.loads(STATUS_PATH.read_text()) if STATUS_PATH.exists() else {}
    except Exception:
        return {}

def _save_status(status):
    atomic_write_text(STATUS_PATH, json.dumps(status, indent=2, default=str))

def is_stage_done(name):
    return _load_status().get(name, {}).get("done", False)

def mark_stage_done(name, meta=None):
    status = _load_status()
    status[name] = {"done": True, "timestamp": time.time(), **(meta or {})}
    _save_status(status)

def clear_stage(name):
    """Force a stage to recompute next time it's requested. Epoch checkpoints of its training
    runs are kept (so it would resume them) — use clear_run() to discard those too."""
    status = _load_status(); status.pop(name, None); _save_status(status)
    _MEMO.pop(name, None)
    print(f"Cleared stage '{name}'.")

def clear_run(run_name):
    p = RUNS_DIR / run_name
    if p.exists():
        shutil.rmtree(p); print(f"Deleted epoch checkpoints of run '{run_name}'.")

def ensure_stage(name, compute_fn, load_fn, force=False):
    if not force and name in _MEMO:
        return _MEMO[name]
    if not force and is_stage_done(name):
        try:
            art = load_fn()
            print(f"[resume] '{name}' loaded from Drive.")
            _MEMO[name] = art
            return art
        except Exception as e:
            print(f"[resume] could not load '{name}' ({type(e).__name__}: {e}) — recomputing.")
    print(f"[run] '{name}' — computing ...")
    t0 = time.time()
    art = compute_fn()
    mark_stage_done(name, {"elapsed_sec": round(time.time() - t0, 1)})
    _MEMO[name] = art
    print(f"[run] '{name}' done in {time.time()-t0:.0f}s — saved to Drive.")
    return art

# --- typed save/load helpers rooted at CKPT_DIR ---------------------------
def save_state_dict(model, name):   atomic_torch_save(model.state_dict(), CKPT_DIR / f"{name}.pt")
def load_state_dict_into(model, name, map_location=None):
    model.load_state_dict(torch_load(CKPT_DIR / f"{name}.pt", map_location)); return model
def save_full_model(model, name):   atomic_torch_save(model, CKPT_DIR / f"{name}.pt")   # for pruned / quantized graphs
def load_full_model(name, map_location=None): return torch_load(CKPT_DIR / f"{name}.pt", map_location)
def save_json(obj, name, root=None): atomic_write_text(Path(root or CKPT_DIR) / f"{name}.json", json.dumps(obj, indent=2, default=str))
def load_json(name, root=None): return json.loads((Path(root or CKPT_DIR) / f"{name}.json").read_text())
def try_load_json(name, root=None):
    p = Path(root or CKPT_DIR) / f"{name}.json"; return json.loads(p.read_text()) if p.exists() else None
def save_df(df, name, root=None): df.to_csv(Path(root or CKPT_DIR) / f"{name}.csv", index=False)
def try_load_df(name, root=None):
    p = Path(root or CKPT_DIR) / f"{name}.csv"; return pd.read_csv(p) if p.exists() else None

print("Checkpoint framework ready.")

---
# S0 · Configuration
Run every session. Two switches at the top:
* `MODE`:
  * `"colab"`: the full Q2 protocol (default)
  * `"lean"`: DIST/CTKD on the main seed only and an 8-epoch recovery budget, about 20 % cheaper
  * `"fast"`: 64 px, short schedules, for an end-to-end check
  * `"smoke"`: synthetic data, only for testing the code
* `DATASET`: `"bloodmnist"` or `"raabin"`. Run the notebook once per dataset.

Seeds vary *training* only: the teacher and the data split are fixed per dataset (`split_seed`). The main seed (`cfg.seed`) is used for the pruning sensitivity analysis, the deployment exports, Grad-CAM and latency.

In [ ]:
MODE = os.environ.get("FCN_MODE", "colab")            # "colab" (full Q2) | "lean" (~20 % cheaper) | "fast" | "smoke"
DATASET = os.environ.get("FCN_DATASET", "bloodmnist")  # "bloodmnist" | "raabin"   ← run the notebook once per dataset

@dataclass
class Config:
    # --- data ---
    dataset: str = "bloodmnist"
    img_size: int = 224
    batch_size: int = 64
    num_workers: int = 2
    train_subset: int = 0                 # >0 caps split sizes (smoke test only)
    seed: int = 42                        # main seed: pruning parent, deployment, XAI, export
    label_smoothing: float = 0.0
    raabin_root: str = "/content/drive/MyDrive/Raabin-WBC"   # unzipped "Double-labeled Cropped Cells"
    raabin_val_frac: float = 0.15         # stratified val split carved from Raabin Train
    split_seed: int = 42                  # data split is fixed across training seeds

    # --- teachers ---
    teacher_arch: str = "convnext_tiny"
    teacher_pretrained: bool = True
    teacher_epochs: int = 10
    teacher_lr: float = 3e-4
    run_vit_teacher: bool = False
    vit_arch: str = "vit_small_patch14_dinov2.lvd142m"
    vit_epochs: int = 8
    vit_lr: float = 5e-5
    kd_teacher: str = "convnext"

    # --- students / RQ1 ---
    student_epochs: int = 15
    student_lr: float = 1e-3
    core_variants: tuple = ("nokd", "logit_kd", "dkd", "dist", "ctkd", "wsld", "adaptive_kd")  # all seeds
    ablation_variants: tuple = ("feat_kd", "multi_kd")                                          # main seed only
    run_ls_kd: bool = False               # optional: logit standardization (CVPR'24) on main seed

    # --- proposed: difficulty-aware multi-level KD ---
    kd_T: float = 4.0
    kd_alpha: float = 0.7
    feat_beta: float = 0.5
    rel_beta: float = 10.0
    diff_alpha: float = 1.0
    diff_beta: float = 1.0
    diff_gamma: float = 1.0
    diff_warmup_epochs: int = 1
    adaptive_mode: str = "bucket"
    w_easy: float = 0.5
    w_hard: float = 1.5

    # --- baselines (paper defaults, NOT tuned — same treatment as the proposed method) ---
    dkd_alpha: float = 1.0                # DKD, Zhao et al. CVPR'22 (CIFAR setting)
    dkd_beta: float = 8.0
    dkd_T: float = 4.0
    dkd_warmup_frac: float = 1 / 12       # paper: 20 of 240 epochs
    dist_beta: float = 2.0                # DIST, Huang et al. NeurIPS'22 (CIFAR setting)
    dist_gamma: float = 2.0
    dist_T: float = 4.0
    ctkd_t_start: float = 1.0             # CTKD, Li et al. AAAI'23: T = t_start + t_range·sigmoid(θ)
    ctkd_t_range: float = 20.0
    ctkd_ramp_frac: float = 0.25          # cosine curriculum λ: 0 → 1 over this fraction of training
    wsld_alpha: float = 2.25              # WSLD, Zhou et al. ICLR'21 (authors' code)
    wsld_T: float = 4.0
    ls_T: float = 2.0                     # logit standardization base temperature

    # --- RQ2: KD-guided progressive pruning ---
    prune_levels: tuple = (0.4, 0.6, 0.75)    # fraction of prunable channels removed
    prune_n_steps: int = 5                    # pruning increments, one per epoch at the start of recovery
    recovery_epochs: int = 10                 # total budget per (level, method): n_steps + fine-tune
    prune_allocation: str = "sensitivity"     # "sensitivity" (per-channel normalised) | "uniform"
    allocation_ablation: bool = True          # also run "uniform" for adaptive_kd, main seed
    sensitivity_ratio: float = 0.5
    recovery_methods: tuple = ("none", "ce", "logit_kd", "dkd", "wsld", "adaptive_kd")
    select_max_val_drop_pp: float = 1.0

    # --- quantization ---
    qat_epochs: int = 3
    qat_polish_epochs: int = 2
    qat_lr: float = 1e-4
    ptq_calib_batches: int = 8
    lowbit_bits: tuple = (8, 6, 4)

    # --- evaluation ---
    robust_subset: int = 1000
    robust_include_int8: bool = True
    xai_n_images: int = 200
    xai_topk: float = 0.2
    bootstrap_n: int = 2000
    bench_iters: int = 100
    bench_repeats: int = 5
    try_tflite: bool = True

    # --- seeds ---
    run_multiseed: bool = True
    seeds: tuple = (42, 43, 44)

    keep_last_ckpt: bool = False

cfg = Config()
cfg.dataset = DATASET

if MODE == "lean":                       # ~20 % cheaper: DIST/CTKD on the main seed only + 8-epoch recovery budget
    cfg.core_variants = ("nokd", "logit_kd", "dkd", "wsld", "adaptive_kd")
    cfg.ablation_variants = ("feat_kd", "multi_kd", "dist", "ctkd")
    cfg.recovery_epochs, cfg.prune_n_steps = 8, 4
elif MODE == "fast":
    cfg.img_size = 64; cfg.batch_size = 128
    cfg.teacher_epochs = 3; cfg.student_epochs = 6; cfg.recovery_epochs = 4; cfg.prune_n_steps = 2
    cfg.qat_epochs = 1; cfg.qat_polish_epochs = 1; cfg.robust_subset = 500; cfg.xai_n_images = 64
    cfg.bench_repeats = 2; cfg.seeds = (42, 43)
elif MODE == "smoke":
    cfg.img_size = 64; cfg.batch_size = 16; cfg.num_workers = 0
    cfg.train_subset = 192; cfg.teacher_pretrained = False
    cfg.teacher_epochs = 1; cfg.student_epochs = 2; cfg.recovery_epochs = 2; cfg.prune_n_steps = 1
    cfg.prune_levels = (0.4, 0.75); cfg.qat_epochs = 1; cfg.qat_polish_epochs = 1; cfg.ptq_calib_batches = 2
    cfg.robust_subset = 64; cfg.xai_n_images = 8; cfg.bootstrap_n = 50
    cfg.bench_iters = 5; cfg.bench_repeats = 2; cfg.seeds = (42, 43); cfg.run_ls_kd = True
    cfg.raabin_root = os.environ.get("FCN_RAABIN_ROOT", cfg.raabin_root)
    if cfg.dataset == "bloodmnist":
        cfg.dataset = "synthetic"

SEEDS = tuple(cfg.seeds) if cfg.run_multiseed else (cfg.seed,)
assert cfg.seed in SEEDS
set_seed(cfg.seed)
set_data_root(cfg.dataset)
print(f"MODE = {MODE!r} | dataset = {cfg.dataset} | seeds = {SEEDS}")
print(json.dumps({k: str(v) for k, v in asdict(cfg).items()}, indent=1))

_cfg_hash = hashlib.sha256(json.dumps(asdict(cfg), sort_keys=True, default=str).encode()).hexdigest()[:12]
_cfg_marker = DATA_ROOT / "config_fingerprint.json"
if _cfg_marker.exists():
    _prev = json.loads(_cfg_marker.read_text())
    if _prev.get("hash") != _cfg_hash:
        print(f"\nNOTE: cfg differs from the one that wrote existing checkpoints (prev={_prev.get('hash')}, now={_cfg_hash}).")
        print("Stages whose settings did not change (teacher, seed-42 RQ1 students) are reused; v2 stages have new names.")
    _hist = _prev.get("history", []) + [{"hash": _cfg_hash, "mode": MODE, "time": time.time()}]
    _prev["history"] = _hist[-20:]
    atomic_write_text(_cfg_marker, json.dumps(_prev, indent=2, default=str))
else:
    atomic_write_text(_cfg_marker, json.dumps({"hash": _cfg_hash, "mode": MODE, "config": asdict(cfg),
                                               "history": [{"hash": _cfg_hash, "mode": MODE, "time": time.time()}]},
                                              indent=2, default=str))
print("Config fingerprint:", _cfg_hash)

## Status dashboard
Run any time: finished stages (load instantly) and training runs (with epochs completed — partially-finished runs will resume).

In [ ]:
def show_status():
    status = _load_status()
    if status:
        rows = [{"stage": k, "done": v.get("done"), "elapsed_sec": v.get("elapsed_sec", ""),
                 "saved_at": pd.to_datetime(v.get("timestamp", 0), unit="s")} for k, v in status.items()]
        display(pd.DataFrame(rows).sort_values("saved_at").reset_index(drop=True))
    else:
        print("No stages completed yet in", DRIVE_ROOT)
    runs = []
    for rd in sorted(RUNS_DIR.glob("*")):
        if not rd.is_dir():
            continue
        h = rd / "history.csv"
        n = len(pd.read_csv(h)) if h.exists() else 0
        steps = len(list(rd.glob("step*_model.pt")))
        runs.append({"run": rd.name, "epochs_done": n if not steps else "", "pruning_steps_saved": steps or "",
                     "finished": (rd / "done.json").exists() or bool(steps),
                     "resumable": (rd / "last.pt").exists()})
    if runs:
        display(pd.DataFrame(runs))

show_status()
print("clear_stage('name') → recompute stage | clear_run('run') → also discard its epoch checkpoints")

---
# S1 · Data — BloodMNIST / Raabin-WBC loaders + corrupted test suite
Loaders are cheap to rebuild each session (only the data card is saved). The training set is wrapped so every batch also carries the **sample index** — the difficulty estimator uses it to look up each image's per-epoch difficulty.

**Loaders:** `train_loader` (augmented, indexed) · `train_clean_loader` (no augmentation, fixed order — teacher-prob cache, difficulty passes, PTQ calibration) · `val_loader` (all selection decisions) · `test_loader` (final reporting only).

**Corruptions (RQ3):** Gaussian noise, Gaussian blur, brightness, contrast, JPEG, colour (hue) shift, resolution reduction — 3 severities each, deterministic per image (seeded by index), applied to the raw image before resizing/normalisation.

**Raabin-WBC.** Set `cfg.raabin_root` to the unzipped *Double-labeled Cropped Cells* folder on Drive. The loader detects the layout itself:
* **Folder names:** it finds `Train` / `TestA` / `TestB` at any depth up to 3, ignoring case, `-`, `_` and spaces.
* **Validation split:** a stratified 15 % of Train, fixed by `split_seed`.
* **Test sets:** TestA is the test set. TestB, if present, is used only as an **external acquisition-shift** test in S11.
* **Flat folders:** if the folder only has class subfolders, it splits them stratified 70/15/15.
* **Cache:** images are resized once per session into a local uint8 memmap (`/content/fcn_cache`), so training never re-decodes 575 px JPEGs.
* **Imbalance:** neutrophils outnumber basophils roughly 30:1, so the class-balanced sampler switches on automatically; report macro-F1 alongside accuracy.

In [ ]:
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image, ImageFilter, ImageEnhance

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.3): self.std, self.p = std, p
    def __call__(self, t):
        return (t + torch.randn_like(t) * self.std).clamp(0, 1) if random.random() < self.p else t

def build_transforms(img_size, train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.RandomRotation(180),
            transforms.ColorJitter(0.15, 0.15, 0.10, 0.03),
            transforms.ToTensor(), AddGaussianNoise(0.02, 0.3),
            transforms.Normalize(MEAN, STD)])
    return transforms.Compose([transforms.Resize((img_size, img_size)),
                               transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

# ---- raw sources: return (PIL image, int label) -----------------------------
class RawMedMNIST(Dataset):
    def __init__(self, split, size):
        from medmnist import BloodMNIST
        try:
            self.ds = BloodMNIST(split=split, download=True, size=size)
        except TypeError:
            self.ds = BloodMNIST(split=split, download=True)
        self.labels = np.asarray(self.ds.labels).reshape(-1).astype(int)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        img, _ = self.ds[i]
        return img.convert("RGB"), int(self.labels[i])

class RawSynthetic(Dataset):
    """Smoke-test only: class-conditional coloured blobs so the pipeline has learnable signal."""
    def __init__(self, split, n, size, num_classes=8):
        rng = np.random.default_rng({"train": 0, "val": 1, "test": 2}[split])
        self.labels = rng.integers(0, num_classes, n)
        yy, xx = np.mgrid[0:size, 0:size]
        self.imgs = []
        for c in self.labels:
            base = rng.uniform(0.6, 0.9, 3)
            img = np.ones((size, size, 3)) * base
            cx, cy = rng.uniform(0.3, 0.7, 2) * size
            r = size * (0.12 + 0.03 * c)
            mask = (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2
            col = np.array([(c * 37 % 255), (c * 91 % 255), (c * 53 % 255)]) / 255.0
            img[mask] = 0.5 * img[mask] + 0.5 * col
            img += rng.normal(0, 0.03, img.shape)
            self.imgs.append((np.clip(img, 0, 1) * 255).astype(np.uint8))
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return Image.fromarray(self.imgs[i]), int(self.labels[i])

class TransformDataset(Dataset):
    def __init__(self, raw, tf, with_index=False, indices=None):
        self.raw, self.tf, self.with_index = raw, tf, with_index
        self.indices = np.arange(len(raw)) if indices is None else np.asarray(indices)
        self.labels = np.asarray(raw.labels)[self.indices]
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, y = self.raw[int(self.indices[i])]
        x = self.tf(img)
        return (x, y, i) if self.with_index else (x, y)

# ---- corruptions ------------------------------------------------------------
CORRUPTIONS = ["gaussian_noise", "gaussian_blur", "brightness", "contrast", "jpeg", "color_shift", "resolution"]
_SEV = {
    "gaussian_noise": [0.04, 0.08, 0.12],      # std on [0,1] pixels
    "gaussian_blur":  [1.0, 2.0, 3.0],         # radius in px at 224 (scaled with image size)
    "brightness":     [0.75, 0.60, 0.45],      # darker illumination
    "contrast":       [0.70, 0.50, 0.30],
    "jpeg":           [40, 20, 8],             # JPEG quality
    "color_shift":    [0.03, 0.06, 0.10],      # hue shift (stain variation)
    "resolution":     [2, 3, 4],               # down-sample factor, then up-sample back
}

def corrupt(img, name, sev, rng):
    """img: PIL RGB. sev in 1..3. Returns PIL RGB."""
    v = _SEV[name][sev - 1]
    W, H = img.size
    if name == "gaussian_noise":
        a = np.asarray(img).astype(np.float32) / 255.0
        a = np.clip(a + rng.normal(0, v, a.shape), 0, 1)
        return Image.fromarray((a * 255).astype(np.uint8))
    if name == "gaussian_blur":
        return img.filter(ImageFilter.GaussianBlur(radius=v * W / 224))
    if name == "brightness":
        return ImageEnhance.Brightness(img).enhance(v)
    if name == "contrast":
        return ImageEnhance.Contrast(img).enhance(v)
    if name == "jpeg":
        buf = io.BytesIO(); img.save(buf, format="JPEG", quality=int(v)); buf.seek(0)
        return Image.open(buf).convert("RGB")
    if name == "color_shift":
        return TF.adjust_hue(img, v)
    if name == "resolution":
        small = img.resize((max(1, W // v), max(1, H // v)), Image.BILINEAR)
        return small.resize((W, H), Image.BILINEAR)
    raise ValueError(name)

class CorruptedDataset(Dataset):
    def __init__(self, raw, indices, name, sev, tf):
        self.raw, self.indices, self.name, self.sev, self.tf = raw, np.asarray(indices), name, sev, tf
        self.labels = np.asarray(raw.labels)[self.indices]
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        j = int(self.indices[i])
        img, y = self.raw[j]
        if self.name != "clean":
            rng = np.random.default_rng(cfg.seed * 1_000_003 + j * 31 + self.sev)
            img = corrupt(img, self.name, self.sev, rng)
        return self.tf(img), y

# ---- Raabin-WBC: auto-detect splits, cache resized uint8 arrays locally -----------
import re
from concurrent.futures import ThreadPoolExecutor
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
CACHE_DIR = Path(os.environ.get("FCN_CACHE", "/content/fcn_cache" if IN_COLAB else "./fcn_cache"))

def _norm(s): return re.sub(r"[^a-z0-9]", "", s.lower())

def _class_dirs(d):
    return sorted([c for c in d.iterdir() if c.is_dir() and any(f.suffix.lower() in IMG_EXT for f in c.iterdir())],
                  key=lambda c: _norm(c.name))

def find_raabin_splits(root):
    """Finds Train / TestA / TestB folders (any case / '-' / '_' / ' ') up to 3 levels below root."""
    root = Path(root)
    assert root.exists(), (f"cfg.raabin_root = {root} does not exist. Download the 'Double-labeled Cropped Cells' "
                           "subset from https://raabindata.com/free-data/, unzip it to Google Drive and set cfg.raabin_root.")
    found = {}
    dirs = [root] + sorted(p for p in root.rglob("*") if p.is_dir() and len(p.relative_to(root).parts) <= 3)
    for d in dirs:
        n = _norm(d.name)
        if not _class_dirs(d): continue
        if n in ("train", "training") and "train" not in found: found["train"] = d
        elif n in ("testa", "test1") and "test" not in found: found["test"] = d
        elif n in ("testb", "test2") and "external" not in found: found["external"] = d
        elif n in ("test", "testing") and "test" not in found: found["test"] = d
    if "train" not in found and _class_dirs(root):
        found["all"] = root                       # flat layout: root/<class>/* → stratified 70/15/15
    return found

def _list_files(split_dir, class_names):
    files, labels, skipped = [], [], 0
    for c in _class_dirs(split_dir):
        n = _norm(c.name)
        if n not in class_names: skipped += 1; continue
        for f in sorted(c.iterdir()):
            if f.suffix.lower() in IMG_EXT: files.append(f); labels.append(class_names.index(n))
    if skipped: print(f"  [raabin] {split_dir.name}: skipped {skipped} folder(s) not among training classes")
    return files, np.array(labels, dtype=np.int64)

def _load_resized(path, size):
    with Image.open(path) as im:
        im.draft("RGB", (size * 2, size * 2))
        return np.asarray(im.convert("RGB").resize((size, size), Image.BICUBIC), dtype=np.uint8)

def build_cache(files, labels, tag, size):
    """Resize once per session into a local uint8 memmap (fast loading, no repeated JPEG decoding)."""
    key = hashlib.sha1(("|".join(str(f) for f in files) + f"|{size}").encode()).hexdigest()[:10]
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    arr_p, done_p = CACHE_DIR / f"{tag}_{size}_{key}.npy", CACHE_DIR / f"{tag}_{size}_{key}.done"
    if not done_p.exists():
        t0 = time.time()
        arr = np.lib.format.open_memmap(arr_p, mode="w+", dtype=np.uint8, shape=(len(files), size, size, 3))
        with ThreadPoolExecutor(8) as ex:
            for i, a in enumerate(ex.map(lambda f: _load_resized(f, size), files)):
                arr[i] = a
        arr.flush(); del arr; done_p.write_text("ok")
        print(f"  [cache] {tag}: {len(files)} images → {arr_p.name} ({time.time()-t0:.0f}s)")
    return np.load(arr_p, mmap_mode="r"), labels

class RawArray(Dataset):
    def __init__(self, arr, labels): self.arr, self.labels = arr, np.asarray(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return Image.fromarray(np.ascontiguousarray(self.arr[i])), int(self.labels[i])

def _stratified(labels, frac, seed, idx=None):
    from sklearn.model_selection import train_test_split
    idx = np.arange(len(labels)) if idx is None else idx
    a, b = train_test_split(idx, test_size=frac, stratify=labels[idx], random_state=seed)
    return np.sort(a), np.sort(b)

def raabin_splits():
    found = find_raabin_splits(cfg.raabin_root)
    print("  [raabin] detected:", {k: str(v) for k, v in found.items()})
    src = found.get("train", found.get("all"))
    assert src is not None, f"No Train/TestA folders (or class folders) found under {cfg.raabin_root}"
    classes = [_norm(c.name) for c in _class_dirs(src)]
    f_tr, y_tr = _list_files(src, classes)
    arr_tr, y_tr = build_cache(f_tr, y_tr, "raabin_train" if "train" in found else "raabin_all", cfg.img_size)
    raw_tr = RawArray(arr_tr, y_tr)
    if "train" in found:
        idx_tr, idx_va = _stratified(y_tr, cfg.raabin_val_frac, cfg.split_seed)
        assert "test" in found, "Found Train but no TestA/Test folder."
        f_te, y_te = _list_files(found["test"], classes)
        raw_te = RawArray(*build_cache(f_te, y_te, "raabin_testA", cfg.img_size)); idx_te = None
        split_desc = f"official Train / TestA; val = stratified {cfg.raabin_val_frac:.0%} of Train (seed {cfg.split_seed})"
    else:
        idx_rest, idx_te = _stratified(y_tr, 0.15, cfg.split_seed)
        idx_tr, idx_va = _stratified(y_tr, 0.15 / 0.85, cfg.split_seed, idx_rest)
        raw_te = raw_tr
        split_desc = f"flat folder → stratified 70/15/15 (seed {cfg.split_seed})"
    raw_ext = None
    if "external" in found:
        f_ex, y_ex = _list_files(found["external"], classes)
        raw_ext = RawArray(*build_cache(f_ex, y_ex, "raabin_testB", cfg.img_size))
    return raw_tr, idx_tr, raw_tr, idx_va, raw_te, idx_te, raw_ext, classes, split_desc

# ---- build everything --------------------------------------------------------
def _cap(raw, n, idx=None):
    idx = np.arange(len(raw)) if idx is None else np.asarray(idx)
    if not n or n >= len(idx):
        return None if len(idx) == len(raw) else idx
    rng = np.random.default_rng(cfg.split_seed)
    return np.sort(rng.choice(idx, n, replace=False))

def _raw_splits():
    """→ raw_tr, idx_tr, raw_va, idx_va, raw_te, idx_te, raw_ext, classes, split description"""
    if cfg.dataset == "bloodmnist":
        from medmnist import INFO
        info = INFO["bloodmnist"]
        classes = [v for _, v in sorted(info["label"].items(), key=lambda kv: int(kv[0]))]
        size = 64 if cfg.img_size <= 64 else (128 if cfg.img_size <= 128 else 224)
        return (RawMedMNIST("train", size), None, RawMedMNIST("val", size), None, RawMedMNIST("test", size), None,
                None, classes, "official 7:1:2 (MedMNIST)")
    if cfg.dataset == "raabin":
        return raabin_splits()
    n = cfg.train_subset or 512
    classes = [f"class_{i}" for i in range(8)]
    return (RawSynthetic("train", n, cfg.img_size), None, RawSynthetic("val", max(32, n // 3), cfg.img_size), None,
            RawSynthetic("test", max(32, n // 3), cfg.img_size), None, None, classes, "synthetic")

def _loader(ds, shuffle=False, sampler=None, drop_last=False, bs=None):
    return DataLoader(ds, batch_size=bs or cfg.batch_size, shuffle=shuffle if sampler is None else False,
                      sampler=sampler, num_workers=cfg.num_workers, pin_memory=(DEVICE.type == "cuda"),
                      drop_last=drop_last, persistent_workers=False)

def get_stage0_data(show_plot=True):
    raw_tr, idx_tr, raw_va, idx_va, raw_te, idx_te, raw_ext, classes, split_desc = _raw_splits()
    n = cfg.train_subset
    idx_tr, idx_va, idx_te = _cap(raw_tr, n, idx_tr), _cap(raw_va, n // 3, idx_va), _cap(raw_te, n // 3, idx_te)
    tf_train, tf_eval = build_transforms(cfg.img_size, True), build_transforms(cfg.img_size, False)
    train_ds       = TransformDataset(raw_tr, tf_train, with_index=True, indices=idx_tr)
    train_clean_ds = TransformDataset(raw_tr, tf_eval,  with_index=True, indices=idx_tr)
    val_ds         = TransformDataset(raw_va, tf_eval, indices=idx_va)
    test_ds        = TransformDataset(raw_te, tf_eval, indices=idx_te)
    ext_ds = TransformDataset(raw_ext, tf_eval, indices=_cap(raw_ext, n // 3)) if raw_ext is not None else None
    K = len(classes)

    counts = np.bincount(train_ds.labels, minlength=K)
    imbalance = counts.max() / max(counts.min(), 1)
    sampler = None
    if imbalance > 3:
        w = 1.0 / counts[train_ds.labels]
        sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(w), replacement=True)
    loaders = {
        "train": _loader(train_ds, shuffle=True, sampler=sampler, drop_last=True),
        "train_clean": _loader(train_clean_ds),
        "val": _loader(val_ds), "test": _loader(test_ds),
        "external": _loader(ext_ds) if ext_ds is not None else None,
    }
    print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)} external={len(ext_ds) if ext_ds else 0} "
          f"classes={K} | imbalance {imbalance:.1f}x | balanced sampler: {sampler is not None}")
    display(pd.DataFrame({"class": classes, "train_count": counts, "pct": (100 * counts / counts.sum()).round(2),
                          "val": np.bincount(val_ds.labels, minlength=K), "test": np.bincount(test_ds.labels, minlength=K),
                          **({"external": np.bincount(ext_ds.labels, minlength=K)} if ext_ds else {})}))

    te_labels = test_ds.labels
    if cfg.robust_subset and cfg.robust_subset < len(test_ds):
        from sklearn.model_selection import train_test_split
        robust_pos, _ = train_test_split(np.arange(len(test_ds)), train_size=cfg.robust_subset,
                                         stratify=te_labels, random_state=cfg.split_seed)
        robust_pos = np.sort(robust_pos)
    else:
        robust_pos = np.arange(len(test_ds))
    robust_raw_idx = test_ds.indices[robust_pos]

    data_card = {
        "dataset": cfg.dataset, "classes": classes, "split": split_desc,
        "n_train": len(train_ds), "n_val": len(val_ds), "n_test": len(test_ds),
        "n_external": len(ext_ds) if ext_ds else 0,
        "train_class_counts": counts.tolist(), "imbalance_ratio": round(float(imbalance), 2),
        "balanced_sampler": sampler is not None,
        "image_size": cfg.img_size, "normalization": {"mean": MEAN, "std": STD},
        "robust_subset_size": int(len(robust_raw_idx)),
        "corruptions": {k: v for k, v in _SEV.items()}, "split_seed": cfg.split_seed,
    }
    save_json(data_card, f"data_card_{cfg.dataset}")

    if show_plot:
        j = int(robust_raw_idx[0]); img0, _ = raw_te[j]
        fig, axes = plt.subplots(3, len(CORRUPTIONS) + 1, figsize=(2 * (len(CORRUPTIONS) + 1), 6.2))
        for s in range(3):
            axes[s, 0].imshow(img0); axes[s, 0].set_title("clean" if s == 0 else "", fontsize=9)
            for c, name in enumerate(CORRUPTIONS):
                rng = np.random.default_rng(cfg.seed * 1_000_003 + j * 31 + s + 1)
                axes[s, c + 1].imshow(corrupt(img0, name, s + 1, rng))
                if s == 0: axes[s, c + 1].set_title(name.replace("_", " "), fontsize=9)
            axes[s, 0].set_ylabel(f"severity {s+1}")
        for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
        plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_corruption_examples.png", dpi=200); plt.show()

    return {"train_ds": train_ds, "train_clean_ds": train_clean_ds, "val_ds": val_ds, "test_ds": test_ds,
            "ext_ds": ext_ds, "raw_test": raw_te, "robust_raw_idx": robust_raw_idx, "tf_eval": tf_eval,
            "classes": classes, "num_classes": K, **{f"{k}_loader": v for k, v in loaders.items()}}

DATA = get_stage0_data()
train_loader, train_clean_loader = DATA["train_loader"], DATA["train_clean_loader"]
val_loader, test_loader = DATA["val_loader"], DATA["test_loader"]
external_loader = DATA["external_loader"]   # Raabin TestB (None otherwise)
CLASSES, NUM_CLASSES = DATA["classes"], DATA["num_classes"]
N_TRAIN = len(DATA["train_ds"])

def make_corrupted_loader(name, sev):
    ds = CorruptedDataset(DATA["raw_test"], DATA["robust_raw_idx"], name, sev, DATA["tf_eval"])
    return _loader(ds)

checkpoint_time("S1 data ready")

---
# S2 · Shared training & evaluation utilities
* **`train_resumable`** — the single trainer used by *every* training run. Saves `runs/<run>/last.pt` after every epoch (model, distillation adapters, optimizer, scheduler, AMP scaler, EMA, RNG state, history) and `best.pt` (best **val** macro-F1). Re-running resumes at the next unfinished epoch.
* **`register_eval`** — evaluates a model on the test set **once**, stores `preds/<name>.npz` (labels + probabilities) and `preds/<name>.json` (metrics). All tables, statistics and figures read from these files, so nothing is re-evaluated after a disconnect.
* Metrics: accuracy, macro-F1/precision/recall, one-vs-rest AUC, ECE (15 bins), per-class recall.

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def model_size_mb(model, bits=32):
    return count_params(model) * bits / 8 / 1e6

def disk_size_mb(model):
    buf = io.BytesIO(); torch.save(model.state_dict(), buf); return buf.tell() / 1e6

def expected_calibration_error(probs, y, n_bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    correct = (pred == y).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def full_metrics(y, probs):
    preds = probs.argmax(1)
    m = {"acc": accuracy_score(y, preds),
         "f1_macro": f1_score(y, preds, average="macro", zero_division=0),
         "precision": precision_score(y, preds, average="macro", zero_division=0),
         "recall": recall_score(y, preds, average="macro", zero_division=0),
         "ece": expected_calibration_error(probs, y)}
    try:
        m["auc_ovr"] = roc_auc_score(y, probs, multi_class="ovr", average="macro",
                                     labels=list(range(probs.shape[1])))
    except Exception:
        m["auc_ovr"] = float("nan")
    m["per_class_recall"] = recall_score(y, preds, average=None, labels=list(range(probs.shape[1])),
                                         zero_division=0).round(4).tolist()
    return {k: (float(v) if not isinstance(v, list) else v) for k, v in m.items()}

def _unpack(batch):
    return batch[0], batch[1], (batch[2] if len(batch) > 2 else None)

@torch.no_grad()
def predict(model, loader, device=DEVICE):
    model.eval().to(device)
    logits, ys = [], []
    for batch in loader:
        x, y, _ = _unpack(batch)
        logits.append(model(x.to(device, non_blocking=True)).float().cpu()); ys.append(y)
    return torch.cat(logits), torch.cat(ys).numpy()

def evaluate(model, loader, device=DEVICE):
    logits, y = predict(model, loader, device)
    m = full_metrics(y, torch.softmax(logits, 1).numpy())
    m.pop("per_class_recall")
    return m

def register_eval(name, model, device=DEVICE, loader=None, force=False):
    """Evaluate on TEST once; persist predictions + metrics to Drive; later calls just load."""
    npz, js = PREDS_DIR / f"{name}.npz", PREDS_DIR / f"{name}.json"
    if not force and npz.exists() and js.exists():
        return json.loads(js.read_text())
    logits, y = predict(model, loader or test_loader, device)
    probs = torch.softmax(logits, 1).numpy()
    m = full_metrics(y, probs)
    np.savez_compressed(npz, y=y, probs=probs.astype(np.float32))
    atomic_write_text(js, json.dumps(m, indent=2))
    print(f"[eval] {name}: acc {m['acc']:.4f} | macro-F1 {m['f1_macro']:.4f} | ECE {m['ece']:.4f}")
    return m

def load_preds(name):
    d = np.load(PREDS_DIR / f"{name}.npz"); return d["y"], d["probs"]

# ---------------------------------------------------------------------------
class CELoss(nn.Module):
    """Plain cross-entropy with the same signature as the distillation losses."""
    def forward(self, logits, y, x=None, idx=None):
        return F.cross_entropy(logits.float(), y, label_smoothing=cfg.label_smoothing)
    def rebind_student(self, student): pass
    def close(self): pass

class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for m in self.shadow.modules():             # drop distillation hooks copied from the live model
            m._forward_hooks.clear()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model):
        for s, m in zip(self.shadow.state_dict().values(), model.state_dict().values()):
            if s.dtype.is_floating_point: s.mul_(self.decay).add_(m.detach(), alpha=1 - self.decay)
            else: s.copy_(m)

@torch.no_grad()
def bn_recalibrate(model, loader=None, device=DEVICE, n_batches=50):
    model = model.to(device)
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            m.reset_running_stats(); m.momentum = None
    model.train()
    for i, batch in enumerate(loader or train_clean_loader):
        model(batch[0].to(device))
        if i + 1 >= n_batches: break
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
            m.momentum = 0.1                        # back to normal EMA for any later training
    return model.eval()

def val_metrics(name, model, device=DEVICE):
    """Cached VAL metrics — used for every selection decision (never test)."""
    p = PREDS_DIR / f"{name}__val.json"
    if p.exists():
        return json.loads(p.read_text())
    m = evaluate(model, val_loader, device)
    atomic_write_text(p, json.dumps(m, indent=2))
    return m

def _rng_state():
    return {"py": random.getstate(), "np": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def _set_rng_state(s):
    random.setstate(s["py"]); np.random.set_state(s["np"]); torch.set_rng_state(s["torch"])
    if s.get("cuda") is not None and torch.cuda.is_available():
        try: torch.cuda.set_rng_state_all(s["cuda"])
        except Exception: pass

def train_resumable(run_name, model, loss_fn, epochs, lr, optimizer="adamw", schedule="onecycle",
                    ema_decay=None, difficulty=None, restarts=2, amp=True, grad_clip=5.0,
                    tr_loader=None, va_loader=None, device=DEVICE, select_metric="f1_macro", verbose=True):
    """Epoch-level resumable trainer. Returns (model loaded with best-val weights, history df)."""
    tr_loader, va_loader = tr_loader or train_loader, va_loader or val_loader
    run_dir = RUNS_DIR / run_name; run_dir.mkdir(parents=True, exist_ok=True)
    last_p, best_p, hist_p, done_p = (run_dir / f for f in ("last.pt", "best.pt", "history.csv", "done.json"))
    model.to(device); loss_fn.to(device)
    if epochs <= 0:
        return model, pd.DataFrame()

    if done_p.exists() and best_p.exists():           # finished in an earlier session
        model.load_state_dict(torch_load(best_p, device))
        print(f"[resume-epoch] '{run_name}' already finished — loaded best weights.")
        return model, pd.read_csv(hist_p) if hist_p.exists() else pd.DataFrame()

    params = [p for p in list(model.parameters()) + list(loss_fn.parameters()) if p.requires_grad]
    if optimizer == "adamw":
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.05)
    else:
        opt = torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=1e-4, nesterov=True)
    steps = max(1, epochs * len(tr_loader))
    if schedule == "onecycle":
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=steps, pct_start=0.15)
    elif schedule == "cosine_restarts":
        sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            opt, T_0=max(1, steps // max(1, restarts)), T_mult=1, eta_min=lr * 0.01)
    elif schedule == "linear":
        sched = torch.optim.lr_scheduler.LinearLR(opt, 1.0, 0.05, total_iters=steps)
    else:
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: 1.0)
    use_amp = amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    ema = ModelEMA(model, ema_decay) if ema_decay else None

    start_ep, history, best_score = 0, [], -1.0
    if last_p.exists():
        ck = torch_load(last_p, device)
        if ck.get("epochs") != epochs:
            raise RuntimeError(f"Run '{run_name}' was started with epochs={ck.get('epochs')} but now "
                               f"epochs={epochs}. clear_run('{run_name}') to restart it.")
        model.load_state_dict(ck["model"]); loss_fn.load_state_dict(ck["loss"], strict=False)
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        if ema is not None and ck.get("ema") is not None: ema.shadow.load_state_dict(ck["ema"])
        history, best_score, start_ep = ck["history"], ck["best_score"], ck["epoch"]
        _set_rng_state(ck["rng"])
        print(f"[resume-epoch] '{run_name}': resuming at epoch {start_ep + 1}/{epochs}")

    set_progress = getattr(loss_fn, "set_progress", None)    # schedule-based losses (DKD warm-up, CTKD curriculum)
    nb = len(tr_loader)
    for ep in range(start_ep, epochs):
        if difficulty is not None:
            difficulty.update(model, ep)
        model.train(); running, seen, t0 = 0.0, 0, time.time()
        for bi, batch in enumerate(tr_loader):
            if set_progress is not None:
                set_progress((ep * nb + bi) / max(steps, 1))
            x, y, idx = _unpack(batch)
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out = model(x)
                loss = loss_fn(out, y, x, idx)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update(); sched.step()
            if ema is not None: ema.update(model)
            running += loss.item() * x.size(0); seen += x.size(0)

        eval_model = ema.shadow if ema is not None else model
        vm = evaluate(eval_model, va_loader, device)
        history.append({"epoch": ep + 1, "train_loss": running / max(seen, 1), **vm})
        if vm[select_metric] > best_score:
            best_score = vm[select_metric]
            atomic_torch_save(eval_model.state_dict(), best_p)
        atomic_torch_save({"epoch": ep + 1, "epochs": epochs, "model": model.state_dict(),
                           "loss": loss_fn.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                           "scaler": scaler.state_dict(), "ema": ema.shadow.state_dict() if ema else None,
                           "history": history, "best_score": best_score, "rng": _rng_state()}, last_p)
        pd.DataFrame(history).to_csv(hist_p, index=False)
        if verbose:
            print(f"[{run_name}] ep {ep+1:>3}/{epochs} loss {running/max(seen,1):.4f} "
                  f"val_acc {vm['acc']:.4f} val_f1 {vm['f1_macro']:.4f} ({time.time()-t0:.0f}s)")

    model.load_state_dict(torch_load(best_p, device))
    atomic_write_text(done_p, json.dumps({"best_val_" + select_metric: best_score, "epochs": epochs}))
    if not cfg.keep_last_ckpt and last_p.exists():
        last_p.unlink()
    return model, pd.DataFrame(history)

print("Utilities ready.")

---
# S3 · Teachers  `[stages: teacher_convnext, teacher_vit (optional), teacher_<kind>_train_probs]`
**ConvNeXt-Tiny** is the main teacher. Set `cfg.run_vit_teacher = True` to also fine-tune a **DINOv2-S** (ViT-S/14) teacher — the comparison asks whether a stronger foundation-model teacher transfers more useful morphological knowledge to a ~1.7–2.4 M-param student.

Recorded per teacher: accuracy, macro-F1, per-class recall, AUC, ECE, confusion matrix, params, size (latency is measured in S10).

**Feature tap points** (early / middle / deep) — resolution matched to the student at every level:

| level | ConvNeXt-T | DINOv2-S | FasterCellNet |
|---|---|---|---|
| early (1/8) | `stages.1` (192 ch) | `blocks.3` tokens → 16×16 map | `stages.2` (80 ch) |
| middle (1/16) | `stages.2` (384) | `blocks.7` | `stages.4` (160) |
| deep (1/32) | `stages.3` (768) | `blocks.11` | `stages.6` (320, after CoordAtt) |

ViT token maps are bilinearly resized to the student's resolution at each level.

**Teacher probability cache:** the teacher's softmax on the *un-augmented* training images is computed once and stored on Drive; the difficulty estimator's entropy and confidence terms read from it.
> Caveat: the teacher has seen these training images, so its train-set confidence is optimistic. The per-epoch teacher–student KL term is what keeps the difficulty signal informative. Cross-fitted (k-fold) teacher probabilities would be the stricter alternative.

In [ ]:
import timm

TEACHER_SPECS = {
    "convnext": {"arch": cfg.teacher_arch, "kind": "cnn", "taps": ["stages.1", "stages.2", "stages.3"],
                 "cam_layer": "stages.3", "epochs": cfg.teacher_epochs, "lr": cfg.teacher_lr},
    "vit":      {"arch": cfg.vit_arch, "kind": "vit", "taps": ["blocks.3", "blocks.7", "blocks.11"],
                 "cam_layer": None, "epochs": cfg.vit_epochs, "lr": cfg.vit_lr},
}

def build_teacher(kind, pretrained):
    spec = TEACHER_SPECS[kind]
    kw = {"img_size": cfg.img_size} if spec["kind"] == "vit" else {}
    try:
        return timm.create_model(spec["arch"], pretrained=pretrained, num_classes=NUM_CLASSES, **kw)
    except Exception as e:
        if not pretrained:
            raise
        print(f"Pretrained weights unavailable for {spec['arch']} ({type(e).__name__}); using random init.")
        return timm.create_model(spec["arch"], pretrained=False, num_classes=NUM_CLASSES, **kw)

def _freeze(m):
    m.eval()
    for p in m.parameters(): p.requires_grad_(False)
    return m

def get_teacher(kind="convnext", force=False):
    name = f"teacher_{kind}"
    spec = TEACHER_SPECS[kind]

    def _compute():
        set_seed(cfg.seed)
        t = build_teacher(kind, cfg.teacher_pretrained).to(DEVICE)
        print(f"Teacher {spec['arch']}: {count_params(t)/1e6:.2f} M params")
        t, hist = train_resumable(name, t, CELoss(), spec["epochs"], spec["lr"])
        save_state_dict(t, name); save_df(hist, f"{name}_history")
        metrics = register_eval(name, t)
        return {"model": _freeze(t), "metrics": metrics, "history": hist}

    def _load():
        t = build_teacher(kind, pretrained=False)
        load_state_dict_into(t, name, DEVICE)
        return {"model": _freeze(t.to(DEVICE)), "metrics": register_eval(name, t),
                "history": try_load_df(f"{name}_history")}

    return ensure_stage(name, _compute, _load, force=force)

def get_teacher_train_probs(kind="convnext", force=False):
    name = f"teacher_{kind}_train_probs"
    def _compute():
        t = get_teacher(kind)["model"]
        logits, _ = predict(t, train_clean_loader)
        probs = torch.softmax(logits, 1).numpy().astype(np.float32)
        np.save(CKPT_DIR / f"{name}.npy", probs)
        conf = probs.max(1)
        print(f"Teacher train-set confidence: mean {conf.mean():.4f} | <0.9 on {100*(conf<0.9).mean():.2f}% of images")
        return {"probs": probs}
    def _load():
        return {"probs": np.load(CKPT_DIR / f"{name}.npy")}
    return ensure_stage(name, _compute, _load, force=force)

def plot_confusion(y, probs, title, fname):
    cm = confusion_matrix(y, probs.argmax(1), labels=list(range(NUM_CLASSES)))
    cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
    short = [str(c)[:11] + ("…" if len(str(c)) > 11 else "") for c in CLASSES]
    fig, ax = plt.subplots(figsize=(1.0 * NUM_CLASSES + 2.5, 1.0 * NUM_CLASSES + 1.5))
    im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(short, rotation=45, ha="right")
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(short)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                    color="white" if cmn[i, j] > 0.5 else "black")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    plt.colorbar(im, fraction=0.046); plt.tight_layout()
    plt.savefig(FIGS_DIR / fname, dpi=300); plt.show()

TEACHER_KINDS = ["convnext"] + (["vit"] if cfg.run_vit_teacher else [])
for _k in TEACHER_KINDS:
    _t = get_teacher(_k)
    print(f"{_k}: params {count_params(_t['model'])/1e6:.3f} M | size {model_size_mb(_t['model']):.1f} MB | "
          + " ".join(f"{k}={v:.4f}" for k, v in _t["metrics"].items() if k != "per_class_recall"))
    get_teacher_train_probs(_k)
    plot_confusion(*load_preds(f"teacher_{_k}"), f"Teacher ({TEACHER_SPECS[_k]['arch']})", f"fig_cm_teacher_{_k}.png")

teacher = get_teacher(cfg.kd_teacher)["model"]
checkpoint_time("S3 teachers ready")

---
# S4 · FasterCellNet student architecture
Same architecture as the ICCIT version: **FasterNet-T0** (PConv, `embed_dim=40`, `depths=(1,2,8,2)`, `mlp_ratio=2`, `n_div=4`) + **Coordinate Attention** after stages 2 and 3. CoordAtt receives its static feature-map size in the constructor, so the whole graph stays `torch.fx`-traceable for INT8 QAT.

Stage outputs are the residual stream, whose width is fixed during pruning (only hidden MLP channels and the head are pruned). The three feature tap points therefore keep 80 / 160 / 320 channels at every pruning level, and the distillation adapters stay valid after pruning.

In [ ]:
class PConv(nn.Module):
    """Partial Convolution: conv on dim//n_div channels, identity on the rest."""
    def __init__(self, dim, n_div=4, kernel_size=3):
        super().__init__()
        self.dim_conv = dim // n_div
        self.dim_untouched = dim - self.dim_conv
        self.partial_conv = nn.Conv2d(self.dim_conv, self.dim_conv, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        if getattr(self, "export_mode", False):      # slicing → builtin StridedSlice (uneven split → Flex op in TFLite)
            return torch.cat((self.partial_conv(x[:, :self.dim_conv]), x[:, self.dim_conv:]), dim=1)
        x1, x2 = torch.split(x, [self.dim_conv, self.dim_untouched], dim=1)
        return torch.cat((self.partial_conv(x1), x2), dim=1)

class FasterNetBlock(nn.Module):
    def __init__(self, dim, mlp_ratio=2.0, n_div=4):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.spatial_mixing = PConv(dim, n_div)
        self.mlp = nn.Sequential(nn.Conv2d(dim, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.GELU(),
                                 nn.Conv2d(hidden, dim, 1, bias=False))
    def forward(self, x):
        return x + self.mlp(self.spatial_mixing(x))

class PatchEmbed(nn.Module):
    def __init__(self, in_ch=3, embed_dim=40, patch=4):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, embed_dim, patch, patch, bias=False); self.norm = nn.BatchNorm2d(embed_dim)
    def forward(self, x): return self.norm(self.proj(x))

class PatchMerging(nn.Module):
    def __init__(self, dim, out_dim, patch=2):
        super().__init__()
        self.reduction = nn.Conv2d(dim, out_dim, patch, patch, bias=False); self.norm = nn.BatchNorm2d(out_dim)
    def forward(self, x): return self.norm(self.reduction(x))

class CoordAtt(nn.Module):
    """Hou et al., CVPR 2021. feat_hw = static (H, W): keeps the module torch.fx-traceable (INT8 QAT) and
    lets the directional pools be fixed-kernel AvgPool2d, which exports to ONNX/TFLite with a dynamic batch
    axis (AdaptiveAvgPool2d((None, 1)) does not). Numerically identical; no parameters, so old weights load."""
    def __init__(self, inp, reduction=32, feat_hw=None):
        super().__init__()
        self.feat_hw = feat_hw
        if feat_hw is not None:
            h, w = feat_hw
            self.pool_h = nn.AvgPool2d((1, w)); self.pool_w = nn.AvgPool2d((h, 1))
        else:
            self.pool_h = nn.AdaptiveAvgPool2d((None, 1)); self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, inp // reduction)
        self.conv1 = nn.Conv2d(inp, mip, 1, bias=False); self.bn1 = nn.BatchNorm2d(mip); self.act = nn.Hardswish()
        self.conv_h = nn.Conv2d(mip, inp, 1); self.conv_w = nn.Conv2d(mip, inp, 1)
    def forward(self, x):
        if getattr(self, "export_mode", False):
            # Export-only path (eval mode): conv1/BN/act are per-position, so running them on the two
            # directional descriptors separately equals the concat→split version exactly — but without the
            # concat/split/permute pattern that TFLite converters mishandle. Broadcasting does the rest.
            a_h = torch.sigmoid(self.conv_h(self.act(self.bn1(self.conv1(self.pool_h(x))))))   # B,C,H,1
            a_w = torch.sigmoid(self.conv_w(self.act(self.bn1(self.conv1(self.pool_w(x))))))   # B,C,1,W
            return x * a_h * a_w
        identity = x
        h, w = self.feat_hw if self.feat_hw is not None else (x.shape[2], x.shape[3])
        x_h = self.pool_h(x); x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        y_h, y_w = torch.split(y, [h, w], dim=2)
        return identity * torch.sigmoid(self.conv_h(y_h)) * torch.sigmoid(self.conv_w(y_w.permute(0, 1, 3, 2)))

class FasterCellNet(nn.Module):
    def __init__(self, num_classes, img_size, embed_dim=40, depths=(1, 2, 8, 2), mlp_ratio=2.0, n_div=4,
                 use_ca=True, ca_stages=(2, 3), head_dim=640, in_ch=3):
        super().__init__()
        dims = [embed_dim * (2 ** i) for i in range(len(depths))]
        self.patch_embed = PatchEmbed(in_ch, dims[0], patch=4)
        stages = []
        for i, d in enumerate(depths):
            blocks = [FasterNetBlock(dims[i], mlp_ratio, n_div) for _ in range(d)]
            if use_ca and i in ca_stages:
                hw = max(1, img_size // (4 * (2 ** i)))
                blocks.append(CoordAtt(dims[i], feat_hw=(hw, hw)))
            stages.append(nn.Sequential(*blocks))
            if i < len(depths) - 1:
                stages.append(PatchMerging(dims[i], dims[i + 1]))
        self.stages = nn.Sequential(*stages)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.head_conv = nn.Conv2d(dims[-1], head_dim, 1, bias=False)
        self.head_act = nn.GELU(); self.flatten = nn.Flatten(1)
        self.head = nn.Linear(head_dim, num_classes)
    def forward_features(self, x): return self.stages(self.patch_embed(x))
    def forward(self, x):
        x = self.avgpool(self.forward_features(x))
        return self.head(self.flatten(self.head_act(self.head_conv(x))))

def prepare_for_export(model):
    """Eval-mode copy for ONNX/TFLite: fixed-kernel pools (also for models unpickled from older
    checkpoints) + the concat-free CoordAtt path. Never used for training or FX quantization."""
    model = copy.deepcopy(model).cpu().eval()
    for m in model.modules():
        if isinstance(m, PConv):
            m.export_mode = True
        if isinstance(m, CoordAtt):
            if m.feat_hw is not None and isinstance(m.pool_h, nn.AdaptiveAvgPool2d):
                h, w = m.feat_hw
                m.pool_h, m.pool_w = nn.AvgPool2d((1, w)), nn.AvgPool2d((h, 1))
            m.export_mode = True
    return model

STUDENT_TAPS =["stages.2", "stages.4", "stages.6"]     # early (1/8) · middle (1/16) · deep (1/32)
STUDENT_CAM_LAYER = "stages.6"
STAGE_MODULES = ["stages.0", "stages.2", "stages.4", "stages.6"]

def build_student(use_ca=True):
    return FasterCellNet(NUM_CLASSES, img_size=cfg.img_size, use_ca=use_ca)

_probe = build_student()
print(f"FasterCellNet: {count_params(_probe)/1e6:.3f} M params ({model_size_mb(_probe):.2f} MB fp32)")
with torch.no_grad():
    print("forward OK:", tuple(_probe(torch.randn(2, 3, cfg.img_size, cfg.img_size)).shape))
try:
    from thop import profile
    macs, _ = profile(copy.deepcopy(_probe), inputs=(torch.randn(1, 3, cfg.img_size, cfg.img_size),), verbose=False)
    print(f"FasterCellNet @ {cfg.img_size}px: {macs/1e6:.1f} MMACs")
except Exception as e:
    print("thop unavailable:", e)
del _probe

---
# S5a · Distillation losses and the difficulty estimator

**Per-sample objective** (every term is computed per image, then averaged over the batch):

$$\mathcal{L}_i=(1-\alpha)\,\mathrm{CE}_i+\lambda_{1,i}\,T^2\,\mathrm{KL}\!\left(p_T^{(T)}\,\|\,p_S^{(T)}\right)_i+\lambda_{2,i}\,\tfrac{1}{3}\sum_{\ell}\big\|\mathrm{LN}(A_\ell F^S_{\ell})-\mathrm{LN}(F^T_\ell)\big\|^2_i+\lambda_{3,i}\,\big\|R^T_{i,:}-R^S_{i,:}\big\|^2$$

* **Logit KD** — temperature-scaled KL.
* **Multi-level feature KD** — trainable 1×1 adapters $A_\ell$ map student channels to teacher channels at three depths. Both sides are layer-normalised per sample, so ConvNeXt/ViT activation scales don't dominate.
* **Relational KD** — $R=\hat f\hat f^{\top}$ is the cosine-similarity matrix of globally-pooled deep features within the batch. Each image's row of $R^T$ must be matched by its row of $R^S$.

**Difficulty** (recomputed at the start of every epoch on the un-augmented training set):

$$D(x)=\frac{a\,H_T(x)/\log K+b\,(1-C_T(x))+g\,\widetilde{\mathrm{KL}}(p_T\|p_S)(x)}{a+b+g}$$

Here $\widetilde{\mathrm{KL}}$ is the teacher–student KL divided by its 99th percentile and clipped to [0, 1]. It is off during `diff_warmup_epochs` for students trained from scratch, while the student is still random. Recovery and QAT start from a trained student, so there it is on from the first epoch. Samples are split into **easy / medium / hard** thirds by rank.

**Adaptive weights** $\lambda_{k}=f(D)$ — `adaptive_mode="bucket"` (default):

| bucket | λ₁ logit | λ₂ feature | λ₃ relational |
|---|---|---|---|
| easy | α·w_easy | 0 | 0 |
| medium | α | β_f | 0 |
| hard | α·w_hard | β_f·w_hard | β_r |

`adaptive_mode="continuous"` uses smooth ramps of the difficulty rank $r\in[0,1]$ instead: λ₁ = α(w_easy + (w_hard − w_easy)·r), λ₂ = β_f·w_hard·clip(1.5r − 0.5), λ₃ = β_r·clip(3r − 2).

Standard KD, feature KD and uniform multi-level KD are the **same class** with fixed λ's. Differences between RQ1 rows therefore come only from which terms are active and whether they are adaptive.

Per-epoch difficulty scores are saved to `runs/<run>/difficulty_epXXX.npy`; S14 uses them to plot how samples move between buckets.

### Baselines (`LogitBaselineLoss`): paper formulations, paper-default hyper-parameters, no tuning
| method | loss | defaults (`cfg`) |
|---|---|---|
| **DKD** (Zhao et al., CVPR 2022) | CE + w(t)·(α·TCKD + β·NCKD), linear warm-up w(t) | α = 1, β = 8, T = 4, warm-up = 1/12 of training |
| **DIST** (Huang et al., NeurIPS 2022) | CE + β·T²(1 − ρ_inter) + γ·T²(1 − ρ_intra), Pearson correlations | β = γ = 2, T = 4 |
| **CTKD** (Li et al., AAAI 2023) | (1 − α)CE + α·T²·KL, with a learnable global T = 1 + 20·σ(θ) trained *adversarially* through a gradient-reversal layer, curriculum λ: 0 → 1 (cosine) | ramp = 25 % of training, T initialised at 4 |
| **WSLD** (Zhou et al., ICLR 2021), an existing **sample-adaptive** KD | CE + α·T²·w_i·CE(p_T, p_S), with w_i = 1 − exp(−CE_S,i / CE_T,i) | α = 2.25, T = 4 |
| KD + Logit Standardization (Sun et al., CVPR 2024), optional | KD on per-sample z-scored logits | T = 2 |

All of these were checked numerically against the authors' reference code (identical values). For recovery and QAT, which start from an already-trained student, the schedule-based baselines run at their post-warm-up setting (DKD term fully on, CTKD λ = 1).

**WSLD is the key comparison.** Like the proposed method, it re-weights distillation per sample; it does so from the student/teacher loss ratio, where the proposed method uses a difficulty score combining teacher uncertainty and teacher–student disagreement.

In [ ]:
class FeatureTap:
    """Forward hooks that keep the latest output of named sub-modules."""
    def __init__(self, model, names):
        self.names, self.feats = list(names), {}
        mods = dict(model.named_modules())
        missing = [n for n in self.names if n not in mods]
        assert not missing, f"tap points not found: {missing}"
        self.handles = [mods[n].register_forward_hook(self._mk(n)) for n in self.names]
    def _mk(self, n):
        def hook(m, i, o): self.feats[n] = o[0] if isinstance(o, (tuple, list)) else o
        return hook
    def get(self): return [self.feats[n] for n in self.names]
    def remove(self):
        for h in self.handles: h.remove()
        self.handles, self.feats = [], {}

class TeacherWithTaps:
    """Frozen teacher → (logits, [early, middle, deep] feature maps as B×C×H×W)."""
    def __init__(self, teacher, kind):
        self.model, self.kind = teacher, TEACHER_SPECS[kind]["kind"]
        self.tap = FeatureTap(teacher, TEACHER_SPECS[kind]["taps"])
        self.n_prefix = getattr(teacher, "num_prefix_tokens", 1)
    @torch.no_grad()
    def __call__(self, x):
        logits = self.model(x)
        feats = self.tap.get()
        if self.kind == "vit":
            feats = [self._tokens_to_map(f) for f in feats]
        return logits, feats
    def _tokens_to_map(self, t):
        t = t[:, self.n_prefix:, :]
        B, N, C = t.shape; g = int(round(N ** 0.5))
        return t.transpose(1, 2).reshape(B, C, g, g)
    def remove(self): self.tap.remove()

class DifficultyEstimator:
    """warmup_epochs: epochs with the KL term off — cfg.diff_warmup_epochs for a randomly initialised
    student, 0 for recovery / QAT (the student is already trained, so disagreement is informative at once)."""
    def __init__(self, teacher_probs, save_dir=None, tag="", warmup_epochs=None):
        p = torch.as_tensor(teacher_probs).float().clamp_min(1e-8)
        self.pT, self.N, K = p, p.shape[0], p.shape[1]
        self.H = -(p * p.log()).sum(1) / math.log(K)
        self.C = p.max(1).values
        self.kl = torch.zeros(self.N)
        self.save_dir, self.tag = (Path(save_dir) if save_dir else None), tag
        self.warmup = cfg.diff_warmup_epochs if warmup_epochs is None else warmup_epochs
        self._recompute(epoch=0, save=False, verbose=False)

    @torch.no_grad()
    def update(self, student, epoch):
        if cfg.diff_gamma > 0 and epoch >= self.warmup:
            logits, _ = predict(student, train_clean_loader)
            pS = torch.softmax(logits, 1).clamp_min(1e-8)
            kl = (self.pT * (self.pT.log() - pS.log())).sum(1)
            self.kl = (kl / torch.quantile(kl, 0.99).clamp_min(1e-6)).clamp(0, 1)
        self._recompute(epoch, save=True)

    def _recompute(self, epoch, save, verbose=True):
        g = cfg.diff_gamma if epoch >= self.warmup else 0.0
        a, b = cfg.diff_alpha, cfg.diff_beta
        self.D = (a * self.H + b * (1 - self.C) + g * self.kl) / max(a + b + g, 1e-8)
        order = self.D.argsort()
        self.rank = torch.empty(self.N); self.rank[order] = torch.arange(self.N, dtype=torch.float) / max(self.N - 1, 1)
        self.bucket = (self.rank * 3).floor().clamp(max=2).long()          # 0 easy · 1 medium · 2 hard
        if save and self.save_dir is not None:
            np.save(self.save_dir / f"difficulty_ep{epoch+1:03d}.npy", self.D.numpy().astype(np.float32))
        if not verbose:
            return
        mean_by_bucket = [float(self.D[self.bucket == k].mean()) for k in range(3)]
        print(f"  [difficulty{self.tag}] ep {epoch+1}: D mean {self.D.mean():.4f} | "
              f"easy/med/hard mean D = {mean_by_bucket[0]:.3f}/{mean_by_bucket[1]:.3f}/{mean_by_bucket[2]:.3f}"
              f" | KL term {'on' if g > 0 else 'off'}")

    def lookup(self, idx, device):
        idx = idx.long().cpu()
        return self.bucket[idx].to(device), self.rank[idx].to(device)

class DistillLoss(nn.Module):
    def __init__(self, student, teacher, teacher_kind, use_logit=True, use_feat=False, use_rel=False,
                 adaptive=False, difficulty=None):
        super().__init__()
        self._teacher = [TeacherWithTaps(teacher, teacher_kind)]      # list → not a sub-module (not saved)
        self.use_logit, self.use_feat, self.use_rel = use_logit, use_feat, use_rel
        self.adaptive, self.difficulty = adaptive, difficulty
        self.alpha, self.T = cfg.kd_alpha, cfg.kd_T
        self.stap = FeatureTap(student, STUDENT_TAPS) if (use_feat or use_rel) else None
        self.adapters = nn.ModuleList()
        if use_feat:
            was_training = student.training
            student.eval()
            dev = next(student.parameters()).device
            with torch.no_grad():
                dummy = torch.randn(2, 3, cfg.img_size, cfg.img_size, device=dev)
                student(dummy); s_feats = self.stap.get()
                _, t_feats = self.teacher(dummy.to(next(teacher.parameters()).device))
            student.train(was_training)
            for s, t in zip(s_feats, t_feats):
                self.adapters.append(nn.Conv2d(s.shape[1], t.shape[1], 1, bias=False))

    @property
    def teacher(self): return self._teacher[0]

    def _lambdas(self, idx, B, dev):
        a, bf, br = self.alpha, cfg.feat_beta, cfg.rel_beta
        ones = torch.ones(B, device=dev)
        if not self.adaptive or idx is None or self.difficulty is None:
            return a * ones, bf * ones, br * ones
        bucket, r = self.difficulty.lookup(idx, dev)
        if cfg.adaptive_mode == "bucket":
            w1 = torch.tensor([cfg.w_easy, 1.0, cfg.w_hard], device=dev)[bucket]
            w2 = torch.tensor([0.0, 1.0, cfg.w_hard], device=dev)[bucket]
            w3 = torch.tensor([0.0, 0.0, 1.0], device=dev)[bucket]
        else:
            w1 = cfg.w_easy + (cfg.w_hard - cfg.w_easy) * r
            w2 = cfg.w_hard * (1.5 * r - 0.5).clamp(0, 1)
            w3 = (3 * r - 2).clamp(0, 1)
        return a * w1, bf * w2, br * w3

    def forward(self, logits, y, x, idx=None):
        logits = logits.float(); B = logits.size(0)
        t_logits, t_feats = self.teacher(x)
        t_logits = t_logits.float()
        lam1, lam2, lam3 = self._lambdas(idx, B, logits.device)
        ce = F.cross_entropy(logits, y, reduction="none", label_smoothing=cfg.label_smoothing)
        loss = (1 - self.alpha) * ce
        if self.use_logit:
            kl = F.kl_div(F.log_softmax(logits / self.T, 1), F.softmax(t_logits / self.T, 1),
                          reduction="none").sum(1) * self.T ** 2
            loss = loss + lam1 * kl
        if self.use_feat or self.use_rel:
            s_feats = self.stap.get()
        if self.use_feat:
            per = 0.0
            for adapter, s, t in zip(self.adapters, s_feats, t_feats):
                s2 = adapter(s).float(); t = t.float()
                if t.shape[-2:] != s2.shape[-2:]:
                    t = F.interpolate(t, size=s2.shape[-2:], mode="bilinear", align_corners=False)
                s2 = F.layer_norm(s2, s2.shape[1:]); t = F.layer_norm(t, t.shape[1:])
                per = per + ((s2 - t) ** 2).flatten(1).mean(1)
            loss = loss + lam2 * per / len(self.adapters)
        if self.use_rel:
            fs = F.normalize(s_feats[-1].float().mean((2, 3)), dim=1)
            ft = F.normalize(t_feats[-1].float().mean((2, 3)), dim=1)
            rel = ((fs @ fs.t() - ft @ ft.t()) ** 2).mean(1)
            loss = loss + lam3 * rel
        return loss.mean()

    def rebind_student(self, student):
        """Point the feature taps at a new student object (after a pruning step); adapters are kept."""
        if self.stap is not None:
            self.stap.remove(); self.stap = FeatureTap(student, STUDENT_TAPS)

    def close(self):
        self.teacher.remove()
        if self.stap is not None: self.stap.remove()

# ---------------------------------------------------------------------------
# Logit-based baselines — paper formulations, paper-default hyper-parameters (cfg.*), not tuned.
# progress ∈ [0,1] = fraction of the run's total training; set by train_resumable each step. For
# multi-phase runs (progressive pruning) progress = offset + scale · (progress within the phase).
# ---------------------------------------------------------------------------
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam; return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return -ctx.lam * g, None

class LogitBaselineLoss(nn.Module):
    KINDS = ("dkd", "dist", "ctkd", "wsld", "ls_kd")

    def __init__(self, kind, teacher, teacher_kind):
        super().__init__()
        assert kind in self.KINDS
        self.kind = kind
        self._teacher = [TeacherWithTaps(teacher, teacher_kind)]
        self.progress, self.progress_offset, self.progress_scale = 0.0, 0.0, 1.0
        self.last_T = None
        if kind == "ctkd":   # learnable global temperature, initialised at T = cfg.kd_T
            p0 = (cfg.kd_T - cfg.ctkd_t_start) / cfg.ctkd_t_range
            self.t_param = nn.Parameter(torch.tensor([math.log(p0 / (1 - p0))]))

    @property
    def teacher(self): return self._teacher[0]

    def set_progress(self, p_local):
        self.progress = min(1.0, self.progress_offset + self.progress_scale * p_local)

    @staticmethod
    def _kl(p_t, log_p_s):                      # per-sample KL(p_t || p_s)
        return F.kl_div(log_p_s, p_t, reduction="none").sum(1)

    def forward(self, logits, y, x, idx=None):
        s = logits.float()
        with torch.no_grad():
            t, _ = self.teacher(x)
        t = t.float(); K = s.size(1)
        ce = F.cross_entropy(s, y, reduction="none", label_smoothing=cfg.label_smoothing)

        if self.kind == "dkd":                  # Zhao et al., CVPR 2022
            T = cfg.dkd_T
            gt = F.one_hot(y, K).bool()
            ps, pt = F.softmax(s / T, 1), F.softmax(t / T, 1)
            ps_b = torch.stack([(ps * gt).sum(1), (ps * ~gt).sum(1)], 1).clamp_min(1e-8)
            pt_b = torch.stack([(pt * gt).sum(1), (pt * ~gt).sum(1)], 1).clamp_min(1e-8)
            tckd = (pt_b * (pt_b.log() - ps_b.log())).sum(1) * T ** 2
            nckd = self._kl(F.softmax(t / T - 1000.0 * gt, 1), F.log_softmax(s / T - 1000.0 * gt, 1)) * T ** 2
            warm = min(self.progress / max(cfg.dkd_warmup_frac, 1e-8), 1.0)
            return (ce + warm * (cfg.dkd_alpha * tckd + cfg.dkd_beta * nckd)).mean()

        if self.kind == "dist":                 # Huang et al., NeurIPS 2022
            T = cfg.dist_T
            ys, yt = F.softmax(s / T, 1), F.softmax(t / T, 1)
            def pearson(a, b, dim):
                a = a - a.mean(dim, keepdim=True); b = b - b.mean(dim, keepdim=True)
                return (a * b).sum(dim) / (a.norm(dim=dim) * b.norm(dim=dim) + 1e-8)
            inter = T ** 2 * (1 - pearson(ys, yt, 1)).mean()
            intra = T ** 2 * (1 - pearson(ys, yt, 0)).mean()
            return ce.mean() + cfg.dist_beta * inter + cfg.dist_gamma * intra

        if self.kind == "ctkd":                 # Li et al., AAAI 2023 (global temperature, cosine curriculum)
            r = min(self.progress / max(cfg.ctkd_ramp_frac, 1e-8), 1.0)
            lam = 0.5 * (1 - math.cos(math.pi * r))
            T = cfg.ctkd_t_start + cfg.ctkd_t_range * torch.sigmoid(_GradReverse.apply(self.t_param, lam))
            self.last_T = float(T.detach())
            kl = self._kl(F.softmax(t / T, 1), F.log_softmax(s / T, 1)) * T ** 2
            return ((1 - cfg.kd_alpha) * ce + cfg.kd_alpha * kl).mean()

        if self.kind == "wsld":                 # Zhou et al., ICLR 2021 — sample-wise adaptive weights
            T = cfg.wsld_T
            soft = -(F.softmax(t / T, 1) * F.log_softmax(s / T, 1)).sum(1)
            ce_s = F.cross_entropy(s.detach(), y, reduction="none")
            ce_t = F.cross_entropy(t, y, reduction="none")
            w = 1 - torch.exp(-(ce_s / (ce_t + 1e-7)).clamp_min(0))
            return (ce + cfg.wsld_alpha * T ** 2 * w * soft).mean()

        if self.kind == "ls_kd":                # Sun et al., CVPR 2024 — logit standardization + KD
            T = cfg.ls_T
            zs = (s - s.mean(1, keepdim=True)) / (s.std(1, keepdim=True) + 1e-7)
            zt = (t - t.mean(1, keepdim=True)) / (t.std(1, keepdim=True) + 1e-7)
            kl = self._kl(F.softmax(zt / T, 1), F.log_softmax(zs / T, 1)) * T ** 2
            return ((1 - cfg.kd_alpha) * ce + cfg.kd_alpha * kl).mean()

    def rebind_student(self, student): pass
    def close(self): self.teacher.remove()

KD_VARIANTS = {   # name: (use_logit, use_feat, use_rel, adaptive)
    "nokd":        None,
    "logit_kd":    (True, False, False, False),
    "feat_kd":     (True, True,  False, False),
    "multi_kd":    (True, True,  True,  False),
    "adaptive_kd": (True, True,  True,  True),
    "adaptive_logit_kd": (True, False, False, True),   # used during QAT (FX graph has no feature hooks)
}
KD_VARIANTS.update({k: "baseline" for k in LogitBaselineLoss.KINDS})
KD_LABELS = {"nokd": "No KD", "logit_kd": "Logit KD (Hinton)", "feat_kd": "Logit + Feature KD",
             "multi_kd": "Logit + Feature + Relational KD (uniform)",
             "dkd": "DKD (CVPR'22)", "dist": "DIST (NeurIPS'22)", "ctkd": "CTKD (AAAI'23)",
             "wsld": "WSLD (ICLR'21, sample-adaptive)", "ls_kd": "KD + Logit Std. (CVPR'24)",
             "adaptive_kd": "Difficulty-aware multi-level KD (proposed)"}

def make_loss(variant, student, teacher_kind=None, run_name="", pretrained_student=False):
    """Returns (loss_fn, difficulty_estimator_or_None). pretrained_student=True (recovery / QAT)
    switches the difficulty KL term on from the first epoch."""
    spec = KD_VARIANTS[variant]
    if spec is None:
        return CELoss(), None
    teacher_kind = teacher_kind or cfg.kd_teacher
    if spec == "baseline":
        return LogitBaselineLoss(variant, get_teacher(teacher_kind)["model"], teacher_kind), None
    use_logit, use_feat, use_rel, adaptive = spec
    diff = None
    if adaptive:
        probs = get_teacher_train_probs(teacher_kind)["probs"]
        run_dir = RUNS_DIR / run_name; run_dir.mkdir(parents=True, exist_ok=True)
        diff = DifficultyEstimator(probs, save_dir=run_dir, warmup_epochs=0 if pretrained_student else None)
    loss = DistillLoss(student, get_teacher(teacher_kind)["model"], teacher_kind, use_logit, use_feat,
                       use_rel, adaptive, diff)
    return loss, diff

print("Distillation losses + difficulty estimator ready.")

---
# S5b · RQ1 — Knowledge transfer to the dense student  `[stages: rq1_<variant>_<teacher>_s<seed>]`
**Core variants, run on every seed:**
* No KD
* Logit KD
* DKD
* DIST
* CTKD
* WSLD
* Difficulty-aware multi-level KD

**Ablations, run on the main seed only:**
* Logit + Feature KD
* Logit + Feature + Relational KD with uniform weights. This controls for "more loss terms" versus "difficulty weighting".

Identical architecture, schedule and data order within a seed. Stage names are the same as in v1, so the BloodMNIST seed-42 runs finished in v1 load instead of retraining.

The table reports mean ± std over seeds. Paired significance tests are in S13.

In [ ]:
def student_run_name(variant, seed=None, teacher_kind=None):
    seed = cfg.seed if seed is None else seed
    if variant == "nokd":
        return f"rq1_nokd_s{seed}"
    return f"rq1_{variant}_{teacher_kind or cfg.kd_teacher}_s{seed}"   # names unchanged → earlier runs are reused

def get_student(variant, seed=None, teacher_kind=None, force=False):
    seed = cfg.seed if seed is None else seed
    name = student_run_name(variant, seed, teacher_kind)

    def _compute():
        set_seed(seed)
        student = build_student().to(DEVICE)
        loss_fn, diff = make_loss(variant, student, teacher_kind, run_name=name)
        try:
            student, hist = train_resumable(name, student, loss_fn, cfg.student_epochs, cfg.student_lr,
                                            difficulty=diff)
            if getattr(loss_fn, "last_T", None) is not None:
                save_json({"final_temperature": loss_fn.last_T}, f"{name}_ctkd_T")
        finally:
            loss_fn.close()
        save_state_dict(student, name); save_df(hist, f"{name}_history")
        val_metrics(name, student)
        return {"model": student.eval(), "metrics": register_eval(name, student), "history": hist}

    def _load():
        m = load_state_dict_into(build_student(), name, DEVICE).to(DEVICE).eval()
        val_metrics(name, m)
        return {"model": m, "metrics": register_eval(name, m), "history": try_load_df(f"{name}_history")}

    return ensure_stage(name, _compute, _load, force=force)

RQ1_ORDER = [v for v in ["nokd", "logit_kd", "feat_kd", "multi_kd", "dkd", "dist", "ctkd", "wsld", "ls_kd", "adaptive_kd"]
             if v in cfg.core_variants or v in cfg.ablation_variants or (v == "ls_kd" and cfg.run_ls_kd)]

def rq1_plan():
    """(variant, seed) pairs: core variants on every seed, ablations on the main seed only."""
    plan = []
    for seed in [cfg.seed] + [s for s in SEEDS if s != cfg.seed]:      # main seed first: pruning needs it
        for v in RQ1_ORDER:
            if v in cfg.core_variants or seed == cfg.seed:
                plan.append((v, seed))
    return plan

for v, seed in rq1_plan():
    get_student(v, seed)
    checkpoint_time(f"RQ1 {v} seed {seed}")
if cfg.run_vit_teacher:
    get_student("adaptive_kd", cfg.seed, teacher_kind="vit")

def _seed_metrics(variant, teacher_kind=None, split="test"):
    out = {}
    for s in SEEDS:
        nm = student_run_name(variant, s, teacher_kind)
        p = PREDS_DIR / (f"{nm}.json" if split == "test" else f"{nm}__val.json")
        if p.exists(): out[s] = json.loads(p.read_text())
    return out

def mean_std_table(rows_spec, metrics=("acc", "f1_macro", "auc_ovr", "ece")):
    """rows_spec: list of (label, {seed: metrics}). Returns mean/std per metric + n_seeds."""
    rows = []
    for label, by_seed in rows_spec:
        if not by_seed: continue
        r = {"Variant": label, "Seeds": len(by_seed)}
        for k in metrics:
            v = np.array([m[k] for m in by_seed.values()], dtype=float)
            r[k] = v.mean(); r[k + "_std"] = v.std(ddof=1) if len(v) > 1 else np.nan
        rows.append(r)
    return pd.DataFrame(rows)

def fmt_pm(df, cols=("acc", "f1_macro", "ece"), scale=100, nd=2):
    """'98.54 ± 0.12' strings for paper tables."""
    out = df[["Variant", "Seeds"]].copy()
    for c in cols:
        out[c] = [f"{scale*m:.{nd}f}" + ("" if np.isnan(sd) else f" ± {scale*sd:.{nd}f}")
                  for m, sd in zip(df[c], df[c + "_std"])]
    return out

RQ1_DF = mean_std_table([(KD_LABELS[v], _seed_metrics(v)) for v in RQ1_ORDER] +
                        ([(KD_LABELS["adaptive_kd"] + " — DINOv2 teacher", _seed_metrics("adaptive_kd", "vit"))]
                         if cfg.run_vit_teacher else []))
if len(RQ1_DF):
    RQ1_DF["Δacc vs No KD (pp)"] = 100 * (RQ1_DF["acc"] - RQ1_DF.loc[0, "acc"])
save_df(RQ1_DF, "table_rq1_knowledge_transfer", TABLES_DIR)
save_df(fmt_pm(RQ1_DF), "table_rq1_knowledge_transfer_pm", TABLES_DIR)
display(fmt_pm(RQ1_DF))

_ct = [try_load_json(f"{student_run_name('ctkd', s)}_ctkd_T") for s in SEEDS]
if any(_ct): print("CTKD learned temperature per seed:", [None if c is None else round(c["final_temperature"], 2) for c in _ct])

---
# S6 · RQ2 — KD-guided progressive structured pruning  `[stages: v2_prune_sensitivity, pgp_r<level>_<method>_s<seed>]`
**Parent model.** Each method starts from the dense difficulty-aware student *of the same seed*, so every recovery method begins from the same weights.

**1 · Sensitivity (main seed), normalised per removed channel.** Each stage is pruned alone by `sensitivity_ratio`, followed by BN recalibration. The cost is then (val drop) ÷ (channels removed). Per-stage ratios are allocated ∝ 1/cost (capped at 0.85) so that the channel-weighted mean equals the target. v1 normalised per *stage*, which over-pruned the small but per-channel-expensive stage 2; here the cheap 2,560-channel stage 4 absorbs most of the pruning. `allocation_ablation=True` also runs uniform allocation (the ICCIT setup) for the adaptive method on the main seed.

**2 · Progressive pruning guided by each method's own loss.** Levels are 40 / 60 / 75 % of prunable channels. For each (level, method, seed) the budget is `recovery_epochs` = 10:
* `prune_n_steps` = 5 increments, each followed by 1 epoch of training with *that method's* loss
* 5 epochs of fine-tuning (cosine with restarts, EMA 0.999, best val macro-F1)

Methods:
* `none`: one-shot pruning + BN recalibration, no training (a lower bound)
* `ce`
* `logit_kd`
* `dkd`
* `wsld`
* `adaptive_kd`

The question is therefore *which training signal best guides compression*. The model is saved to Drive after every increment (together with the distillation adapters), so a disconnect resumes at the increment it was on.

In [ ]:
import torch_pruning as tp
print("torch-pruning", tp.__version__)

def _importance():
    for n in ["GroupMagnitudeImportance", "GroupNormImportance", "MagnitudeImportance"]:
        if hasattr(tp.importance, n):
            return getattr(tp.importance, n)(p=1)
    raise RuntimeError("No magnitude importance in torch_pruning.")

def _ignored_layers(model):
    ignored = [model.head, model.patch_embed.proj]
    for m in model.modules():
        if isinstance(m, PatchMerging): ignored.append(m.reduction)
        if isinstance(m, PConv): ignored.append(m.partial_conv)
        if isinstance(m, CoordAtt): ignored += [m.conv1, m.conv_h, m.conv_w]
        if isinstance(m, FasterNetBlock): ignored.append(m.mlp[3])
    return ignored

def _prunable_channels(model):
    """Hidden-MLP channels per stage (the only prunable width inside stages)."""
    mods = dict(model.named_modules())
    return {s: sum(m.mlp[0].out_channels for m in mods[s].modules() if isinstance(m, FasterNetBlock))
            for s in STAGE_MODULES}

def prune_once(model, global_ratio, stage_ratios=None, only_stage=None):
    """One-shot structured pruning of `model` in place."""
    example = torch.randn(1, 3, cfg.img_size, cfg.img_size, device=next(model.parameters()).device)
    ignored = _ignored_layers(model)
    if only_stage is not None:
        ignored += [m for n, m in model.named_modules()
                    if isinstance(m, (nn.Conv2d, nn.Linear)) and not n.startswith(only_stage + ".")]
    ratio_dict = None
    if stage_ratios:
        mods = dict(model.named_modules())
        ratio_dict = {mods[s]: float(max(r, 0.0)) for s, r in stage_ratios.items()}
    kw = dict(importance=_importance(), iterative_steps=1, ignored_layers=ignored)
    try:
        pruner = tp.pruner.MetaPruner(model, example, pruning_ratio=global_ratio, pruning_ratio_dict=ratio_dict, **kw)
    except TypeError:
        pruner = tp.pruner.MetaPruner(model, example, ch_sparsity=global_ratio, ch_sparsity_dict=ratio_dict, **kw)
    model.eval(); pruner.step()
    return model

from collections import OrderedDict

def save_model_no_hooks(model, path):
    """torch.save a full (pruned) model while distillation hooks are attached (closures don't pickle)."""
    saved = [(m, m._forward_hooks) for m in model.modules()]
    for m, _ in saved: m._forward_hooks = OrderedDict()
    try:
        atomic_torch_save(model, path)
    finally:
        for m, h in saved: m._forward_hooks = h

def get_parent_student(seed=None):
    """Pruning parent = the dense difficulty-aware student of the same seed (shared by every recovery method)."""
    return get_student("adaptive_kd", cfg.seed if seed is None else seed)["model"]

def rtag(level): return f"r{int(round(level * 100))}"

# ---- 1. sensitivity (main seed), normalised PER CHANNEL removed ------------------
def get_prune_sensitivity(force=False):
    def _compute():
        parent = get_parent_student()
        base = evaluate(parent, val_loader)["acc"]
        before = _prunable_channels(parent); rows = []
        for s in STAGE_MODULES:
            m = prune_once(copy.deepcopy(parent), cfg.sensitivity_ratio, only_stage=s)
            m = bn_recalibrate(m)
            acc = evaluate(m, val_loader)["acc"]
            removed = before[s] - _prunable_channels(m)[s]
            drop = 100 * (base - acc)
            rows.append({"stage": s, "prunable_channels": before[s], "channels_removed": removed,
                         "val_acc": acc, "val_drop_pp": drop,
                         "drop_per_channel_pp": max(drop, 0.0) / max(removed, 1),
                         "params_removed_pct": 100 * (1 - count_params(m) / count_params(parent))})
            print(f"  {s}: removed {removed}/{before[s]} ch → val acc {acc:.4f} (drop {drop:+.2f} pp, "
                  f"{rows[-1]['drop_per_channel_pp']:.4f} pp/ch)")
        df = pd.DataFrame(rows); save_df(df, "v2_prune_sensitivity"); save_df(df, "table_prune_sensitivity", TABLES_DIR)
        return {"df": df, "base_val_acc": base}
    return ensure_stage("v2_prune_sensitivity", _compute,
                        lambda: {"df": try_load_df("v2_prune_sensitivity"), "base_val_acc": None}, force=force)

def allocate_stage_ratios(target, sens_df, r_max=0.85):
    """Per-stage ratios ∝ 1/(accuracy cost per removed channel); channel-weighted mean = target.
    (v1 normalised per STAGE, which over-pruned the small, per-channel-expensive stage 2.)"""
    C = sens_df["prunable_channels"].to_numpy(float)
    cost = sens_df["drop_per_channel_pp"].to_numpy(float)
    pos = cost[cost > 0]
    eps = 0.1 * (np.median(pos) if len(pos) else 1e-3)
    w = 1.0 / (cost + eps)
    w = w * C.sum() / (w * C).sum()
    r = np.clip(target * w, 0, r_max)
    for _ in range(200):
        deficit = target * C.sum() - (r * C).sum()
        free = r < r_max - 1e-9
        if abs(deficit) / C.sum() < 1e-5 or not free.any(): break
        r[free] += deficit * w[free] / (w[free] * C[free]).sum()
        r = np.clip(r, 0, r_max)
    return {s: float(v) for s, v in zip(sens_df["stage"], r.round(4))}

def _schedule(level, n):
    return [level * (k + 1) / n for k in range(n)]

def _step_ratios(prev, cum, level, final_stage):
    g = 1 - (1 - cum) / (1 - prev)
    if not final_stage:
        return g, None
    return g, {s: 1 - (1 - rf * cum / level) / (1 - rf * prev / level) for s, rf in final_stage.items()}

# ---- 2. KD-guided progressive pruning -------------------------------------------------
RECOVERY_LABELS = {"none": "No recovery (one-shot + BN recal.)", "ce": "CE fine-tune", "logit_kd": "Logit KD",
                   "dkd": "DKD", "dist": "DIST", "ctkd": "CTKD", "wsld": "WSLD (sample-adaptive)",
                   "ls_kd": "KD + Logit Std.", "adaptive_kd": "Adaptive KD (proposed)"}
RECOVERY_TO_VARIANT = {"ce": "nokd"}

def post_warmup(loss_fn):
    """Recovery/QAT start from a trained student → schedule-based baselines run at their post-warm-up
    setting (DKD fully on, CTKD curriculum λ = 1)."""
    if hasattr(loss_fn, "progress_offset"):
        loss_fn.progress_offset, loss_fn.progress_scale = 1.0, 0.0
        loss_fn.set_progress(0.0)

def pgp_name(level, method, seed, allocation=None):
    allocation = allocation or cfg.prune_allocation
    suf = "" if allocation == cfg.prune_allocation else f"_{allocation}"
    return f"pgp_{rtag(level)}_{method}_s{seed}{suf}"

def get_pgp(level, method, seed, allocation=None, force=False):
    allocation = allocation or cfg.prune_allocation
    name = pgp_name(level, method, seed, allocation)

    def _compute():
        parent = get_parent_student(seed)
        final_stage = allocate_stage_ratios(level, get_prune_sensitivity()["df"]) if allocation == "sensitivity" else None
        model = copy.deepcopy(parent).to(DEVICE)
        run_dir = RUNS_DIR / name; run_dir.mkdir(parents=True, exist_ok=True)
        E, n = cfg.recovery_epochs, cfg.prune_n_steps
        assert E > n, "recovery_epochs must exceed prune_n_steps"
        hist = None
        if method == "none":
            model = bn_recalibrate(prune_once(model, level, final_stage))
        else:
            set_seed(seed)
            loss_fn, diff = make_loss(RECOVERY_TO_VARIANT.get(method, method), model, run_name=name,
                                      pretrained_student=True)
            post_warmup(loss_fn)
            try:
                prev = 0.0
                for k, cum in enumerate(_schedule(level, n), 1):
                    sp, lp = run_dir / f"step{k}_model.pt", run_dir / f"step{k}_loss.pt"
                    if sp.exists() and lp.exists():
                        model = torch_load(sp, DEVICE)
                        loss_fn.load_state_dict(torch_load(lp, DEVICE), strict=False)
                        loss_fn.rebind_student(model); prev = cum
                        print(f"  [{name}] step {k} loaded from Drive"); continue
                    g, st = _step_ratios(prev, cum, level, final_stage)
                    model = bn_recalibrate(prune_once(model, g, st))
                    loss_fn.rebind_student(model)
                    model, _ = train_resumable(f"{name}_step{k}", model, loss_fn, 1, cfg.student_lr * 0.1,
                                               optimizer="sgd", schedule="constant", difficulty=diff, verbose=False)
                    save_model_no_hooks(model, sp); atomic_torch_save(loss_fn.state_dict(), lp); prev = cum
                    print(f"  [{name}] step {k}/{n}: cumulative {cum:.2f} → {count_params(model)/1e6:.3f} M params")
                model, hist = train_resumable(f"{name}_ft", model, loss_fn, E - n, cfg.student_lr * 0.05,
                                              optimizer="sgd", schedule="cosine_restarts", restarts=1,
                                              ema_decay=0.999, difficulty=diff)
            finally:
                loss_fn.close()
        model.eval()
        save_full_model(model, name)
        if hist is not None: save_df(hist, f"{name}_history")
        info = {"level": level, "method": method, "seed": seed, "allocation": allocation, "stage_ratios": final_stage,
                "params": count_params(model), "params_removed_pct": 100 * (1 - count_params(model) / count_params(parent)),
                "prunable_channels": _prunable_channels(model)}
        save_json(info, f"{name}_info")
        return {"model": model, "info": info, "metrics": register_eval(name, model), "val": val_metrics(name, model)}

    def _load():
        m = load_full_model(name, DEVICE).eval()
        return {"model": m, "info": load_json(f"{name}_info"), "metrics": register_eval(name, m),
                "val": val_metrics(name, m)}

    return ensure_stage(name, _compute, _load, force=force)

SENS = get_prune_sensitivity()
display(SENS["df"].round(4))
for lv in cfg.prune_levels:
    print(f"level {lv:.0%} allocation:", allocate_stage_ratios(lv, SENS["df"]) if cfg.prune_allocation == "sensitivity" else "uniform")

for seed in [cfg.seed] + [s for s in SEEDS if s != cfg.seed]:
    for lv in cfg.prune_levels:
        for meth in cfg.recovery_methods:
            get_pgp(lv, meth, seed)
        checkpoint_time(f"pruning {lv:.0%} seed {seed}")
if cfg.allocation_ablation:
    alt = "uniform" if cfg.prune_allocation == "sensitivity" else "sensitivity"
    for lv in cfg.prune_levels:
        get_pgp(lv, "adaptive_kd", cfg.seed, allocation=alt)
    checkpoint_time("allocation ablation")

---
# S7 · RQ2 results and operating point  `[stage: v2_prune_selection]`
**Operating point.** The deployment level is the **most-pruned level whose adaptive-KD val accuracy, averaged over seeds, is within `select_max_val_drop_pp` of the dense adaptive student**. It is chosen on val only, and the full Pareto front over all levels and methods is reported anyway.

The RQ2 table gives mean ± std over seeds. The allocation ablation (per-channel sensitivity vs uniform) follows it.

In [ ]:
def get_prune_selection(force=False):
    """Operating point from VAL only: most-pruned level whose adaptive-KD val accuracy (mean over seeds)
    is within cfg.select_max_val_drop_pp of the dense adaptive student (mean over seeds)."""
    def _compute():
        meth = "adaptive_kd" if "adaptive_kd" in cfg.recovery_methods else cfg.recovery_methods[-1]
        dense_val = np.mean([val_metrics(student_run_name("adaptive_kd", s), get_parent_student(s))["acc"] for s in SEEDS])
        cands = []
        for lv in cfg.prune_levels:
            v = np.mean([get_pgp(lv, meth, s)["val"]["acc"] for s in SEEDS])
            cands.append({"level": lv, "val_acc_mean": float(v), "val_drop_pp": float(100 * (dense_val - v))})
        ok = [c for c in cands if c["val_drop_pp"] <= cfg.select_max_val_drop_pp]
        chosen = max(ok, key=lambda c: c["level"]) if ok else min(cands, key=lambda c: c["level"])
        sel = {"level": chosen["level"], "dense_val_acc_mean": float(dense_val), "candidates": cands,
               "rule": f"most-pruned level with mean val drop <= {cfg.select_max_val_drop_pp} pp ({meth}); "
                       "fallback = least-pruned"}
        save_json(sel, "v2_prune_selection")
        return sel
    return ensure_stage("v2_prune_selection", _compute, lambda: load_json("v2_prune_selection"), force=force)

SEL = get_prune_selection()
SEL_LEVEL = SEL["level"]; sel_tag = rtag(SEL_LEVEL)
print(f"Selected operating point: {SEL_LEVEL:.0%} of prunable channels removed — {SEL['rule']}")

def _pgp_seed_metrics(level, method, split="test", allocation=None):
    out = {}
    for s in (SEEDS if allocation in (None, cfg.prune_allocation) else (cfg.seed,)):
        nm = pgp_name(level, method, s, allocation)
        p = PREDS_DIR / (f"{nm}.json" if split == "test" else f"{nm}__val.json")
        if p.exists(): out[s] = json.loads(p.read_text())
    return out

def rq2_table():
    dense = _seed_metrics("adaptive_kd")
    dense_acc = np.mean([m["acc"] for m in dense.values()])
    rows = []
    for lv in cfg.prune_levels:
        info = load_json(f"{pgp_name(lv, 'adaptive_kd' if 'adaptive_kd' in cfg.recovery_methods else cfg.recovery_methods[0], cfg.seed)}_info")
        for meth in cfg.recovery_methods:
            te, va = _pgp_seed_metrics(lv, meth), _pgp_seed_metrics(lv, meth, "val")
            if not te: continue
            acc = np.array([m["acc"] for m in te.values()]); f1 = np.array([m["f1_macro"] for m in te.values()])
            vac = np.array([m["acc"] for m in va.values()])
            rows.append({"Pruned": f"{int(round(lv*100))}%", "level": lv, "Recovery": RECOVERY_LABELS[meth], "method": meth,
                         "Params (M)": info["params"] / 1e6, "Params removed (%)": info["params_removed_pct"],
                         "Seeds": len(te), "Val acc": vac.mean(), "Test acc": acc.mean(),
                         "Test acc std": acc.std(ddof=1) if len(acc) > 1 else np.nan,
                         "Test macro-F1": f1.mean(), "Test macro-F1 std": f1.std(ddof=1) if len(f1) > 1 else np.nan,
                         "Δacc vs dense adaptive (pp)": 100 * (acc.mean() - dense_acc), "Selected": lv == SEL_LEVEL})
    return pd.DataFrame(rows)

RQ2_DF = rq2_table()
save_df(RQ2_DF, "table_rq2_pruning_recovery", TABLES_DIR)
RQ2_PM = RQ2_DF[["Pruned", "Recovery", "Params (M)", "Seeds"]].copy()
RQ2_PM["Test acc (%)"] = [f"{100*m:.2f} ± {100*s:.2f}" if not np.isnan(s) else f"{100*m:.2f}"
                       for m, s in zip(RQ2_DF["Test acc"], RQ2_DF["Test acc std"])]
RQ2_PM["Macro-F1 (%)"] = [f"{100*m:.2f} ± {100*s:.2f}" if not np.isnan(s) else f"{100*m:.2f}"
                       for m, s in zip(RQ2_DF["Test macro-F1"], RQ2_DF["Test macro-F1 std"])]
save_df(RQ2_PM, "table_rq2_pruning_recovery_pm", TABLES_DIR)
display(RQ2_PM)

# allocation ablation (main seed)
if cfg.allocation_ablation:
    alt = "uniform" if cfg.prune_allocation == "sensitivity" else "sensitivity"
    abl = []
    for lv in cfg.prune_levels:
        for alloc in (cfg.prune_allocation, alt):
            nm = pgp_name(lv, "adaptive_kd", cfg.seed, alloc)
            if (PREDS_DIR / f"{nm}.json").exists():
                info = load_json(f"{nm}_info"); m = json.loads((PREDS_DIR / f"{nm}.json").read_text())
                abl.append({"Pruned": f"{int(round(lv*100))}%", "Allocation": alloc, "Params (M)": info["params"] / 1e6,
                            "Val acc": json.loads((PREDS_DIR / f"{nm}__val.json").read_text())["acc"],
                            "Test acc": m["acc"], "Stage ratios": info["stage_ratios"]})
    ALLOC_DF = pd.DataFrame(abl); save_df(ALLOC_DF, "table_allocation_ablation", TABLES_DIR); display(ALLOC_DF)

fig, ax = plt.subplots(figsize=(7, 4.2))
for meth in cfg.recovery_methods:
    d = RQ2_DF[RQ2_DF["method"] == meth].sort_values("level")
    if not len(d): continue
    ax.errorbar(d["Params (M)"], 100 * d["Test acc"], yerr=100 * d["Test acc std"].fillna(0), marker="o",
                capsize=3, lw=2.2 if meth == "adaptive_kd" else 1.2, label=RECOVERY_LABELS[meth])
ax.axhline(100 * np.mean([m["acc"] for m in _seed_metrics("adaptive_kd").values()]), ls="--", c="gray",
           label="dense adaptive-KD student")
ax.set_xlabel("Params (M)"); ax.set_ylabel("Test accuracy (%) — mean ± std over seeds")
ax.set_title(f"RQ2 — KD-guided progressive pruning ({cfg.dataset})"); ax.invert_xaxis(); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_rq2_recovery.png", dpi=300); plt.show()

---
# S8 · Quantization — INT8 PTQ / QAT + low-bit simulation  `[stages: int8_*, v2_lowbit_sim]`
At the selected level, **on every seed**:

| row | INT8 recipe | loss during QAT |
|---|---|---|
| No recovery | **PTQ** (calibration only) | — |
| Logit KD | QAT + polish | logit KD |
| **Best baseline** (DKD / WSLD / …, chosen by mean *val* accuracy at this level) | QAT + polish | its own loss |
| **Adaptive KD (proposed)** | QAT + polish | difficulty-aware logit KD |

PTQ of the adaptive model (main seed) and a dense INT8 model (main seed, for deployment) are added as references. The FX-QAT graph has no forward hooks, so the proposed method keeps only its difficulty-weighted logit term during QAT.

**Low-bit (INT6/INT4):** weight-only fake quantization of the selected pruned FP32 model. These rows are labelled **simulated**.

In [ ]:
from torch.ao.quantization import get_default_qat_qconfig_mapping, get_default_qconfig_mapping, disable_observer
from torch.ao.quantization.quantize_fx import prepare_qat_fx, prepare_fx, convert_fx
from torch.ao.nn.intrinsic.qat import freeze_bn_stats

BACKEND = "x86" if "x86" in torch.backends.quantized.supported_engines else "fbgemm"
torch.backends.quantized.engine = BACKEND
EXAMPLE_INPUTS = (torch.randn(1, 3, cfg.img_size, cfg.img_size),)
print("Quantization backend:", BACKEND)

def _prepare(model, qat):
    qmap = get_default_qat_qconfig_mapping(BACKEND) if qat else get_default_qconfig_mapping(BACKEND)
    fn = prepare_qat_fx if qat else prepare_fx
    m = copy.deepcopy(model).cpu(); m.train() if qat else m.eval()
    try:
        return fn(m, qmap, EXAMPLE_INPUTS), "full-graph"
    except Exception as e:
        print(f"  FX trace failed ({type(e).__name__}) — CoordAtt kept as float leaf.")
        from torch.ao.quantization.fx.custom_config import PrepareCustomConfig
        pcc = PrepareCustomConfig().set_non_traceable_module_classes([CoordAtt])
        m = copy.deepcopy(model).cpu(); m.train() if qat else m.eval()
        return fn(m, qmap, EXAMPLE_INPUTS, prepare_custom_config=pcc), "CA-in-float"

def quant_coverage(model):
    q = f = 0
    for m in model.modules():
        if list(m.children()): continue
        mod = type(m).__module__
        if "quantized" in mod or "qat" in mod: q += 1
        else: f += 1
    return {"quantized_leaf": q, "fp32_leaf": f, "coverage_pct": 100 * q / max(q + f, 1)}

# loss used during QAT for each recovery method (FX graph has no feature hooks → adaptive uses its logit term)
QAT_LOSS = {"none": "nokd", "ce": "nokd", "logit_kd": "logit_kd", "adaptive_kd": "adaptive_logit_kd",
            "dkd": "dkd", "dist": "dist", "ctkd": "ctkd", "wsld": "wsld", "ls_kd": "ls_kd"}

def get_int8(source_name, source_model_fn, recipe, method, force=False):
    """source_name: FP32 model id (e.g. pgp_r60_adaptive_kd_s42). recipe: 'ptq' | 'qat'."""
    name = f"int8_{source_name}_{recipe}"

    def _compute():
        src = source_model_fn().eval()
        if recipe == "ptq":
            prepared, mode = _prepare(src, qat=False)
            with torch.no_grad():
                for i, batch in enumerate(train_clean_loader):
                    prepared(batch[0].cpu())
                    if i + 1 >= cfg.ptq_calib_batches: break
        else:
            prepared, mode = _prepare(src, qat=True)
            loss_fn, diff = make_loss(QAT_LOSS[method], prepared, run_name=f"{name}_A", pretrained_student=True)
            post_warmup(loss_fn)
            try:
                prepared, _ = train_resumable(f"{name}_A", prepared, loss_fn, cfg.qat_epochs, cfg.qat_lr,
                                              optimizer="sgd", schedule="linear", amp=False, difficulty=diff)
                prepared.apply(disable_observer); prepared.apply(freeze_bn_stats)
                prepared, _ = train_resumable(f"{name}_B", prepared, loss_fn, cfg.qat_polish_epochs, cfg.qat_lr * 0.1,
                                              optimizer="sgd", schedule="linear", amp=False, difficulty=diff)
            finally:
                loss_fn.close()
            mode += "+QAT+polish"
        prepared.eval().cpu()
        # Converted INT8 FX graphs don't unpickle reliably → persist the calibrated / QAT-trained *prepared*
        # state (weights + observer / fake-quant stats) and rebuild + convert on load (bit-exact).
        atomic_torch_save(prepared.state_dict(), CKPT_DIR / f"{name}_prepared_state.pt")
        q = convert_fx(prepared)
        cov = quant_coverage(q)
        info = {"source": source_name, "recipe": recipe, "method": method, "mode": mode, **cov,
                "params_fp32": count_params(src), "size_mb": disk_size_mb(q)}
        print(f"  [{name}] {mode} | INT8 leaf coverage {cov['coverage_pct']:.1f}% | {info['size_mb']:.2f} MB")
        save_json(info, f"{name}_info")
        return {"model": q, "info": info, "metrics": register_eval(name, q, device=CPU)}

    def _load():
        prepared, _ = _prepare(source_model_fn().eval(), qat=(recipe == "qat"))
        prepared.load_state_dict(torch_load(CKPT_DIR / f"{name}_prepared_state.pt", "cpu"))
        if recipe == "qat":
            prepared.apply(disable_observer); prepared.apply(freeze_bn_stats)
        q = convert_fx(prepared.eval().cpu())
        return {"model": q, "info": load_json(f"{name}_info"), "metrics": register_eval(name, q, device=CPU)}

    return ensure_stage(name, _compute, _load, force=force)

def best_baseline_at(level):
    """Strongest non-proposed KD baseline at `level` by mean VAL accuracy (never test)."""
    cands = [m for m in cfg.recovery_methods if m not in ("none", "ce", "logit_kd", "adaptive_kd")]
    if not cands: return None
    scores = {m: np.mean([v["acc"] for v in _pgp_seed_metrics(level, m, "val").values()]) for m in cands}
    return max(scores, key=scores.get)

BEST_BASELINE = best_baseline_at(SEL_LEVEL)
QAT_METHODS = [m for m in ("logit_kd", BEST_BASELINE, "adaptive_kd") if m and m in cfg.recovery_methods]
print(f"QAT at {SEL_LEVEL:.0%}: methods {QAT_METHODS} (best baseline by val: {BEST_BASELINE}); PTQ for 'none'")

INT8_RUNS = {}   # (source_name, recipe) -> stage artefacts
for seed in [cfg.seed] + [s for s in SEEDS if s != cfg.seed]:
    if "none" in cfg.recovery_methods:
        src = pgp_name(SEL_LEVEL, "none", seed)
        INT8_RUNS[(src, "ptq")] = get_int8(src, (lambda sd=seed: get_pgp(SEL_LEVEL, "none", sd)["model"]), "ptq", "none")
    for meth in QAT_METHODS:
        src = pgp_name(SEL_LEVEL, meth, seed)
        fn = (lambda m=meth, sd=seed: get_pgp(SEL_LEVEL, m, sd)["model"])
        INT8_RUNS[(src, "qat")] = get_int8(src, fn, "qat", meth)
        if seed == cfg.seed and meth == "adaptive_kd":
            INT8_RUNS[(src, "ptq")] = get_int8(src, fn, "ptq", meth)
    checkpoint_time(f"INT8 seed {seed}")
_dense = student_run_name("adaptive_kd")
INT8_RUNS[(_dense, "qat")] = get_int8(_dense, get_parent_student, "qat", "adaptive_kd")
INT8_RUNS[(_dense, "ptq")] = get_int8(_dense, get_parent_student, "ptq", "adaptive_kd")

# ---- low-bit (weight-only) simulation -----------------------------------------
@torch.no_grad()
def fake_quant_weights(model, bits):
    m = copy.deepcopy(model).eval(); qmax = 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)):
            w = mod.weight.data; flat = w.reshape(w.shape[0], -1)
            scale = (flat.abs().max(1).values / qmax).clamp_min(1e-12).view(-1, *[1] * (w.dim() - 1))
            mod.weight.data = (w / scale).round().clamp(-qmax - 1, qmax) * scale
    return m

def get_lowbit_sim(force=False):
    def _compute():
        meth = "adaptive_kd" if "adaptive_kd" in cfg.recovery_methods else "none"
        src_name = pgp_name(SEL_LEVEL, meth, cfg.seed)
        src = get_pgp(SEL_LEVEL, meth, cfg.seed)["model"]
        rows = []
        for b in cfg.lowbit_bits:
            nm = f"lowbit_w{b}_{src_name}"
            met = register_eval(nm, fake_quant_weights(src, b))
            rows.append({"Model": f"W{b}A32 (simulated)", "source": src_name, "bits": b,
                         "est_weight_size_MB": count_params(src) * b / 8 / 1e6, **{k: met[k] for k in ["acc", "f1_macro", "ece"]}})
        df = pd.DataFrame(rows); save_df(df, "v2_lowbit_sim"); save_df(df, "table_lowbit_simulation", TABLES_DIR)
        return {"df": df}
    return ensure_stage("v2_lowbit_sim", _compute, lambda: {"df": try_load_df("v2_lowbit_sim")}, force=force)

LOWBIT = get_lowbit_sim()

QUANT_DF = pd.DataFrame([{**{k: v["info"][k] for k in ("source", "recipe", "method", "mode", "coverage_pct", "params_fp32", "size_mb")},
                          "acc": v["metrics"]["acc"], "f1_macro": v["metrics"]["f1_macro"], "ece": v["metrics"]["ece"]}
                         for v in INT8_RUNS.values()])
save_df(QUANT_DF, "table_quantization", TABLES_DIR)
display(QUANT_DF.round(4)); display(LOWBIT["df"].round(4))
checkpoint_time("S8 quantization done")

---
# S9 · Export — ONNX (FP32 / INT8) and TFLite (FP32 / full-integer INT8)  `[stages: export_*]`
Deployment set, matching the on-device plan:

| id | model |
|---|---|
| `dense` | dense adaptive-KD FasterCellNet |
| `pruned` | selected pruned + adaptive-KD-recovered FasterCellNet |

For each model the notebook exports:
* **ONNX FP32**, with a parity check against PyTorch (max |Δlogit| and top-1 agreement on 256 val images)
* **ONNX INT8** — ONNX Runtime static QDQ quantization, per-channel, calibrated on training images
* **TFLite FP32 and full-integer INT8** via `onnx2tf`, with the representative dataset drawn from training images

Every exported model's **test accuracy is measured with its own runtime**, so the on-device numbers come from the exact files that ship.

**Export-friendly graph.** `prepare_for_export()` changes three things; the logits are the same (verified diff = 0):
* PConv's uneven `split` becomes slicing
* CoordAtt's concat → split → permute becomes two broadcast branches with the same weights, and its pools become fixed-kernel `AvgPool2d`
* ONNX opset 20 is used, so GELU is a native `Gelu` op rather than `Erf`

Together these keep the TFLite graph free of *Flex* (select-TF) ops, so full-integer INT8 converts and runs with the stock TFLite runtime and delegates. Training and FX-QAT still use the original forward.

> The TFLite/ONNX INT8 files are **post-training quantized** from the FP32 checkpoint. PyTorch FX QAT weights don't convert directly to TFLite; `ai-edge-torch` with PT2E-QAT is the route if QAT weights are needed on-device. This is why S14 reports the PyTorch-QAT and TFLite-INT8 accuracies separately.

The TFLite step installs TensorFlow tooling and can fail on version drift. If it does, the stage is **not** marked done, the error is printed, and the ONNX files are still usable. Re-run after fixing the packages.

In [ ]:
import onnx, onnxruntime as ort

_meth_dep = "adaptive_kd" if "adaptive_kd" in cfg.recovery_methods else "none"
DEPLOY = {
    "dense":  (student_run_name("adaptive_kd"), get_parent_student),
    "pruned": (pgp_name(SEL_LEVEL, _meth_dep, cfg.seed), lambda: get_pgp(SEL_LEVEL, _meth_dep, cfg.seed)["model"]),
}

def _val_batch(n=256):
    xs = []
    for batch in val_loader:
        xs.append(batch[0])
        if sum(len(x) for x in xs) >= n: break
    return torch.cat(xs)[:n]

def _calib_images(n):
    xs = []
    for batch in train_clean_loader:
        xs.append(batch[0])
        if sum(len(x) for x in xs) >= n: break
    return torch.cat(xs)[:n]

def ort_predict(path, loader, nchw=True):
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    iname = sess.get_inputs()[0].name
    outs, ys = [], []
    for batch in loader:
        outs.append(sess.run(None, {iname: batch[0].numpy().astype(np.float32)})[0]); ys.append(batch[1].numpy())
    return np.concatenate(outs), np.concatenate(ys)

def export_onnx(model, path):
    model = prepare_for_export(model)
    x = torch.randn(1, 3, cfg.img_size, cfg.img_size)
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=20,
              dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
    try:
        torch.onnx.export(model, (x,), str(path), dynamo=False, **kw)
    except TypeError:                                   # older torch without the `dynamo` kwarg
        torch.onnx.export(model, (x,), str(path), **kw)
    except Exception as e:                              # legacy exporter failed → torch.export-based exporter
        print(f"  legacy ONNX exporter failed ({type(e).__name__}); retrying with dynamo=True")
        kw.pop("dynamic_axes")
        torch.onnx.export(model, (x,), str(path), dynamo=True, external_data=False,
                          dynamic_shapes={"x": {0: torch.export.Dim("batch", min=1, max=1024)}}, **kw)
    onnx.checker.check_model(onnx.load(str(path)))

def _eval_logits(logits, y):
    m = full_metrics(y, torch.softmax(torch.as_tensor(logits).float(), 1).numpy()); m.pop("per_class_recall"); return m

def get_export_onnx(key, force=False):
    src_name, fn = DEPLOY[key]
    name = f"v2_export_onnx_{key}"
    def _compute():
        from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType
        model = fn().eval()
        fp32 = EXPORT_DIR / f"fastercellnet_{key}_{src_name}_fp32.onnx"
        export_onnx(model, fp32)
        # parity vs PyTorch
        xv = _val_batch()
        with torch.no_grad(): ref = model.cpu()(xv).numpy()
        model.to(DEVICE)
        sess = ort.InferenceSession(str(fp32), providers=["CPUExecutionProvider"])
        got = sess.run(None, {"input": xv.numpy()})[0]
        parity = {"max_abs_logit_diff": float(np.abs(ref - got).max()),
                  "top1_agreement": float((ref.argmax(1) == got.argmax(1)).mean())}
        # ORT static INT8 (QDQ, per-channel)
        calib = _calib_images(cfg.ptq_calib_batches * cfg.batch_size).numpy().astype(np.float32)
        class _Reader(CalibrationDataReader):
            def __init__(self): self.it = iter([{"input": calib[i:i + 1]} for i in range(len(calib))])
            def get_next(self): return next(self.it, None)
        int8 = EXPORT_DIR / f"fastercellnet_{key}_{src_name}_int8_ort.onnx"
        quantize_static(str(fp32), str(int8), _Reader(), quant_format=QuantFormat.QDQ, per_channel=True,
                        activation_type=QuantType.QInt8, weight_type=QuantType.QInt8)
        res = {"source": src_name, "fp32_path": str(fp32), "int8_path": str(int8),
               "fp32_size_mb": os.path.getsize(fp32) / 1e6, "int8_size_mb": os.path.getsize(int8) / 1e6,
               "parity_fp32": parity}
        for tag, p in (("onnx_fp32", fp32), ("onnx_int8", int8)):
            logits, y = ort_predict(p, test_loader)
            res[f"{tag}_test"] = _eval_logits(logits, y)
            np.savez_compressed(PREDS_DIR / f"{tag}_{key}.npz", y=y,
                                probs=torch.softmax(torch.as_tensor(logits), 1).numpy().astype(np.float32))
        save_json(res, name, EXPORT_DIR)
        print(f"[{key}] ONNX fp32 parity: {parity} | test acc fp32 {res['onnx_fp32_test']['acc']:.4f} "
              f"int8 {res['onnx_int8_test']['acc']:.4f} | sizes {res['fp32_size_mb']:.2f}/{res['int8_size_mb']:.2f} MB")
        return res
    return ensure_stage(name, _compute, lambda: load_json(name, EXPORT_DIR), force=force)

# ---- TFLite via onnx2tf ---------------------------------------------------------
def _ensure_onnx2tf():
    try:
        import onnx2tf  # noqa
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx2tf", "onnx_graphsurgeon", "sng4onnx",
                        "tf_keras", "psutil", "ai_edge_litert", "onnxsim"], check=False)

def _tflite_interpreter(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf; Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path)); it.allocate_tensors(); return it

def tflite_predict(path, loader, limit=None):
    it = _tflite_interpreter(path)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    outs, ys, n = [], [], 0
    for batch in loader:
        for x, y in zip(batch[0].numpy(), batch[1].numpy()):
            a = np.transpose(x, (1, 2, 0))[None].astype(np.float32)                 # NCHW → NHWC
            if inp["dtype"] != np.float32:
                s, z = inp["quantization"]; a = np.clip(np.round(a / s + z), -128, 127).astype(inp["dtype"])
            it.set_tensor(inp["index"], a); it.invoke()
            o = it.get_tensor(out["index"]).astype(np.float32)
            if out["dtype"] != np.float32:
                s, z = out["quantization"]; o = (o - z) * s
            outs.append(o[0]); ys.append(int(y)); n += 1
            if limit and n >= limit: return np.stack(outs), np.array(ys)
    return np.stack(outs), np.array(ys)

def get_export_tflite(key, force=False):
    name = f"v2_export_tflite_{key}"
    def _compute():
        _ensure_onnx2tf()
        onnx_meta = get_export_onnx(key)
        out_dir = EXPORT_DIR / f"tflite_{key}_{onnx_meta['source']}"; out_dir.mkdir(exist_ok=True)
        calib = _calib_images(max(64, cfg.ptq_calib_batches * cfg.batch_size))
        raw = (calib * torch.tensor(STD).view(1, 3, 1, 1) + torch.tensor(MEAN).view(1, 3, 1, 1)).clamp(0, 1)
        calib_path = out_dir / "calib_nhwc.npy"
        np.save(calib_path, raw.permute(0, 2, 3, 1).numpy().astype(np.float32))
        # onnx2tf's own output check otherwise downloads 20 sample images — give it ours instead (offline-safe)
        smp = F.interpolate(raw[:20], size=(128, 128), mode="bilinear", align_corners=False)
        np.save(out_dir / "calibration_image_sample_data_20x128x128x3_float32.npy",
                smp.permute(0, 2, 3, 1).numpy().astype(np.float32))
        cmd = ["onnx2tf", "-i", onnx_meta["fp32_path"], "-o", str(out_dir), "-b", "1", "-oiqt", "-qt", "per-channel",
               "-cind", "input", str(calib_path), json.dumps([[[MEAN]]]), json.dumps([[[STD]]])]
        r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(out_dir))
        if r.returncode != 0:
            raise RuntimeError("onnx2tf failed:\n" + r.stdout[-3000:] + r.stderr[-3000:])
        f32 = sorted(out_dir.glob("*_float32.tflite")); i8 = sorted(out_dir.glob("*_full_integer_quant.tflite"))
        if not f32 or not i8:
            raise RuntimeError("onnx2tf ran but did not write float32 + full_integer_quant files:\n" + r.stdout[-3000:])
        files = {"tflite_fp32": f32[0], "tflite_int8": i8[0]}
        res = {"source": onnx_meta["source"]}
        xv = _val_batch(64)          # parity reference = ONNX fp32 (itself checked against PyTorch)
        sess = ort.InferenceSession(onnx_meta["fp32_path"], providers=["CPUExecutionProvider"])
        ref_logits = sess.run(None, {"input": xv.numpy()})[0]
        for tag, p in files.items():
            lv, _ = tflite_predict(p, [(xv, torch.zeros(len(xv)))])
            logits, y = tflite_predict(p, test_loader)
            res[tag] = {"path": str(p), "size_mb": os.path.getsize(p) / 1e6,
                        "top1_agreement_vs_onnx_fp32": float((lv.argmax(1) == ref_logits.argmax(1)).mean()),
                        "max_abs_logit_diff_vs_onnx_fp32": float(np.abs(lv - ref_logits).max()),
                        "test": _eval_logits(logits, y)}
            np.savez_compressed(PREDS_DIR / f"{tag}_{key}.npz", y=y,
                                probs=torch.softmax(torch.as_tensor(logits), 1).numpy().astype(np.float32))
            print(f"[{key}] {tag}: {res[tag]['size_mb']:.2f} MB | test acc {res[tag]['test']['acc']:.4f} | "
                  f"agreement vs ONNX fp32 {res[tag]['top1_agreement_vs_onnx_fp32']:.3f}")
        save_json(res, name, EXPORT_DIR)
        return res
    return ensure_stage(name, _compute, lambda: load_json(name, EXPORT_DIR), force=force)

EXPORTS = {}
for key in DEPLOY:
    EXPORTS[key] = {"onnx": get_export_onnx(key)}
    if not cfg.try_tflite:
        EXPORTS[key]["tflite"] = None; continue
    try:
        EXPORTS[key]["tflite"] = get_export_tflite(key)
    except Exception as e:
        EXPORTS[key]["tflite"] = None
        print(f"[{key}] TFLite export failed — not marked done, re-run this cell after fixing: "
              f"{type(e).__name__}: {str(e)[:1500]}")
checkpoint_time("S9 export done")

---
# S10 · Host-CPU latency  `[cached per model in bench/]`
Batch-1 latency on this runtime's CPU, with threads pinned: `cfg.bench_repeats` × `cfg.bench_iters` timed runs after warm-up. The table reports the mean of the repeat means, the std across repeats, and p95 over all runs. The CPU model is recorded with every entry, because Colab CPUs vary between sessions. Smartphone numbers come from S16.

In [ ]:
BENCH_THREADS = min(4, os.cpu_count() or 1)
try:
    CPU_NAME = next((l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor())
except Exception:
    CPU_NAME = platform.processor()

def _timeit(run, warmup=10):
    for _ in range(warmup): run()
    means, allt = [], []
    for _ in range(cfg.bench_repeats):
        ts = []
        for _ in range(cfg.bench_iters):
            t0 = time.perf_counter(); run(); ts.append((time.perf_counter() - t0) * 1000)
        means.append(np.mean(ts)); allt += ts
    means = np.array(means)
    return {"latency_mean_ms": float(means.mean()), "latency_std_ms": float(means.std(ddof=1)) if len(means) > 1 else 0.0,
            "latency_p95_ms": float(np.percentile(allt, 95)), "throughput_ips": float(1000 / means.mean()),
            "cpu": CPU_NAME, "threads": BENCH_THREADS}

def bench(name, kind, obj, force=False):
    """kind: 'torch' (nn.Module, run on CPU) | 'onnx' (path) | 'tflite' (path). Cached in bench/<name>.json."""
    p = BENCH_DIR / f"{name}.json"
    if p.exists() and not force:
        return json.loads(p.read_text())
    torch.set_num_threads(BENCH_THREADS)
    x = torch.randn(1, 3, cfg.img_size, cfg.img_size)
    if kind == "torch":
        m = copy.deepcopy(obj).cpu().eval()
        with torch.no_grad():
            r = _timeit(lambda: m(x))
    elif kind == "onnx":
        so = ort.SessionOptions(); so.intra_op_num_threads = BENCH_THREADS
        sess = ort.InferenceSession(str(obj), so, providers=["CPUExecutionProvider"])
        xin = {sess.get_inputs()[0].name: x.numpy()}
        r = _timeit(lambda: sess.run(None, xin))
    else:
        it = _tflite_interpreter(obj); d = it.get_input_details()[0]
        a = np.zeros(d["shape"], dtype=d["dtype"])
        def _run(): it.set_tensor(d["index"], a); it.invoke()
        r = _timeit(_run)
    atomic_write_text(p, json.dumps(r, indent=2))
    print(f"[bench] {name}: {r['latency_mean_ms']:.2f} ± {r['latency_std_ms']:.2f} ms (p95 {r['latency_p95_ms']:.2f})")
    return r

# Latency depends on architecture, not on training → one representative model per architecture.
BENCH = {}
BENCH[f"teacher_{cfg.kd_teacher}"] = bench(f"teacher_{cfg.kd_teacher}", "torch", teacher)
_dn = student_run_name("adaptive_kd"); BENCH[_dn] = bench(_dn, "torch", get_parent_student())
for lv in cfg.prune_levels:
    nm = pgp_name(lv, _meth_dep, cfg.seed); BENCH[nm] = bench(nm, "torch", get_pgp(lv, _meth_dep, cfg.seed)["model"])
for (src, recipe), art in INT8_RUNS.items():
    if src.endswith(f"_s{cfg.seed}") or src == _dn:
        nm = f"int8_{src}_{recipe}"; BENCH[nm] = bench(nm, "torch", art["model"])
for key, ex in EXPORTS.items():
    src = ex["onnx"]["source"]
    for tag, path in (("onnx_fp32", ex["onnx"]["fp32_path"]), ("onnx_int8", ex["onnx"]["int8_path"])):
        BENCH[f"{tag}_{key}"] = bench(f"v2_{tag}_{src}", "onnx", path)
    if ex.get("tflite"):
        for tag in ("tflite_fp32", "tflite_int8"):
            BENCH[f"{tag}_{key}"] = bench(f"v2_{tag}_{src}", "tflite", ex["tflite"][tag]["path"])
display(pd.DataFrame(BENCH).T.round(3))
checkpoint_time("S10 benchmark done")

---
# S11 · RQ3 — Robustness to corruption and distribution shift  `[robustness/robustness_results.csv, resumable]`
Dense and pruned FP32 models on **every seed** and the INT8 models on the main seed are all evaluated on the same stratified test subset (`cfg.robust_subset`; 0 = full test set): clean, plus 7 corruptions × 3 severities. The loop runs **corruption-major**, so each corrupted image is generated once and fed to every model. Results are appended to Drive after each (corruption, severity) set, and a disconnect resumes at the next unfinished set.

$$\Delta A = A_{\text{clean}}-A_{\text{corrupted}}\qquad \text{Robustness retention}=\frac{A_{\text{model, corrupted}}}{A_{\text{teacher, corrupted}}}$$

reported per corruption, and as means over all 21 sets (mCA = mean corrupted accuracy).

**Raabin-WBC TestB** (if present) is evaluated in full as a *natural* acquisition shift, reported as `external_acc`.

Tables report the mean over seeds per method; `table_rq3_robustness_per_model.csv` keeps every seed.

In [ ]:
def robustness_models():
    """name -> (group label, seed, model, device). Dense + pruned FP32 on every seed; INT8 (CPU) on the main seed."""
    ms = {f"teacher_{cfg.kd_teacher}": ("Teacher", None, teacher, DEVICE)}
    for v in cfg.core_variants:
        for s in SEEDS:
            ms[student_run_name(v, s)] = (f"Dense · {KD_LABELS[v]}", s, get_student(v, s)["model"], DEVICE)
    for meth in cfg.recovery_methods:
        for s in SEEDS:
            ms[pgp_name(SEL_LEVEL, meth, s)] = (f"Pruned {int(round(SEL_LEVEL*100))}% FP32 · {RECOVERY_LABELS[meth]}", s,
                                                get_pgp(SEL_LEVEL, meth, s)["model"], DEVICE)
    if cfg.robust_include_int8:
        for (src, recipe), art in INT8_RUNS.items():
            info = art["info"]
            if src.startswith("pgp_") and src.endswith(f"_s{cfg.seed}") and (recipe == "qat" or info["method"] == "none"):
                ms[f"int8_{src}_{recipe}"] = (f"Pruned INT8-{recipe.upper()} · {RECOVERY_LABELS[info['method']]}",
                                              cfg.seed, art["model"], CPU)
    return ms

ROBUST_CSV = ROBUST_DIR / "robustness_results_v2.csv"

def run_robustness():
    models = robustness_models()
    cols = ["model", "group", "seed", "corruption", "severity", "acc", "f1_macro", "n"]
    done = pd.read_csv(ROBUST_CSV) if ROBUST_CSV.exists() else pd.DataFrame(columns=cols)
    have = set(zip(done["model"], done["corruption"], done["severity"].astype(int)))
    sets = [("clean", 0)] + [(c, s) for c in CORRUPTIONS for s in (1, 2, 3)]
    if external_loader is not None:
        sets.append(("external", 0))                 # Raabin TestB: natural acquisition shift (full set)
    for c, s in sets:
        pending = [n for n in models if (n, c, s) not in have]
        if not pending: continue
        t0 = time.time()
        loader = external_loader if c == "external" else make_corrupted_loader(c, s)
        preds = {n: [] for n in pending}; ys = []
        with torch.no_grad():
            for x, y in loader:
                ys.append(y.numpy())
                for n in pending:
                    _, _, m, dev = models[n]
                    preds[n].append(m.eval().to(dev)(x.to(dev)).float().argmax(1).cpu().numpy())
        y = np.concatenate(ys); rows = []
        for n in pending:
            p = np.concatenate(preds[n]); grp, sd, _, _ = models[n]
            rows.append({"model": n, "group": grp, "seed": sd, "corruption": c, "severity": s,
                         "acc": accuracy_score(y, p), "f1_macro": f1_score(y, p, average="macro", zero_division=0), "n": len(y)})
        new = pd.DataFrame(rows)
        done = pd.concat([done, new], ignore_index=True) if len(done) else new
        tmp = ROBUST_CSV.with_suffix(".tmp"); done.to_csv(tmp, index=False); os.replace(tmp, ROBUST_CSV)
        print(f"  {c:>15} s{s}: {len(pending)} models in {time.time()-t0:.0f}s")
    return pd.read_csv(ROBUST_CSV), models

ROB_RAW, ROB_MODELS = run_robustness()
ROB_RAW = ROB_RAW[ROB_RAW["model"].isin(ROB_MODELS)]

def robustness_summary(df):
    tname = f"teacher_{cfg.kd_teacher}"
    clean = df[df.corruption == "clean"].set_index("model")["acc"]
    ext = df[df.corruption == "external"].set_index("model")["acc"]
    cor = df[~df.corruption.isin(["clean", "external"])].copy()
    t_acc = cor[cor.model == tname].set_index(["corruption", "severity"])["acc"]
    cor["delta_A_pp"] = 100 * (cor["model"].map(clean) - cor["acc"])
    cor["retention"] = cor["acc"].values / t_acc.reindex(list(zip(cor.corruption, cor.severity))).values
    per_model = cor.groupby(["model", "group"]).agg(mCA=("acc", "mean"), mean_delta_A_pp=("delta_A_pp", "mean"),
                                                    worst_acc=("acc", "min"), mean_retention=("retention", "mean")).reset_index()
    per_model["clean_acc"] = per_model["model"].map(clean)
    per_model["external_acc"] = per_model["model"].map(ext) if len(ext) else np.nan
    agg = per_model.groupby("group", sort=False).agg(
        seeds=("model", "count"),
        clean_acc=("clean_acc", "mean"), mCA=("mCA", "mean"), mCA_std=("mCA", "std"),
        mean_delta_A_pp=("mean_delta_A_pp", "mean"), worst_acc=("worst_acc", "mean"),
        mean_retention=("mean_retention", "mean"), external_acc=("external_acc", "mean"),
        external_acc_std=("external_acc", "std")).reset_index()
    order = list(dict.fromkeys(g for g, *_ in ROB_MODELS.values()))
    agg = agg.set_index("group").loc[[g for g in order if g in set(agg.group)]].reset_index()
    per_corr = cor.groupby(["group", "corruption"])["acc"].mean().unstack()
    return agg, per_model, per_corr

ROB_SUMMARY, ROB_PER_MODEL, ROB_PER_CORR = robustness_summary(ROB_RAW)
save_df(ROB_SUMMARY, "table_rq3_robustness_summary", TABLES_DIR)
save_df(ROB_PER_MODEL, "table_rq3_robustness_per_model", TABLES_DIR)
save_df(ROB_PER_CORR.reset_index(), "table_rq3_robustness_per_corruption", TABLES_DIR)
display(ROB_SUMMARY.round(4)); display(ROB_PER_CORR.round(4))

lbl_order = [g for g in ROB_SUMMARY["group"] if g in ROB_PER_CORR.index]
H = ROB_PER_CORR.loc[lbl_order, CORRUPTIONS] * 100
fig, ax = plt.subplots(figsize=(1.1 * len(CORRUPTIONS) + 4.5, 0.42 * len(H) + 1.8))
im = ax.imshow(H.values, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(CORRUPTIONS))); ax.set_xticklabels([c.replace("_", "\n") for c in CORRUPTIONS], fontsize=8)
ax.set_yticks(range(len(H))); ax.set_yticklabels(H.index, fontsize=7)
for i in range(H.shape[0]):
    for j in range(H.shape[1]):
        ax.text(j, i, f"{H.values[i, j]:.1f}", ha="center", va="center", fontsize=6.5,
                color="white" if H.values[i, j] < np.nanmean(H.values) else "black")
plt.colorbar(im, label="accuracy (%) — mean over severities and seeds")
ax.set_title(f"RQ3 — accuracy under corruption ({cfg.dataset})")
plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_rq3_robustness_heatmap.png", dpi=300); plt.show()
checkpoint_time("S11 robustness done")

---
# S12 · Explainability — does the student look where the teacher looks?  `[stage: xai_gradcam]`
Main seed. Grad-CAM on the teacher's last stage (`stages.3`) and on each student's deep stage (`stages.6`, after CoordAtt). Both CAMs target the **teacher's predicted class**, so the maps answer the same question. They are upsampled to the input size and min–max normalised per image.

Overlap metrics per image, averaged over `cfg.xai_n_images` stratified test images:
* **Pearson r** between the two CAMs
* **IoU of the top-k % regions** (`cfg.xai_topk`, default the top 20 % of pixels)

INT8 graphs have no autograd, so the selected **pruned FP32** model stands in for the deployed INT8 model. This is noted in the table.

In [ ]:
def gradcam(model, layer_name, x, target):
    """Batch Grad-CAM. Returns (B, H, W) maps in [0, 1] at input resolution."""
    model.eval()
    layer = dict(model.named_modules())[layer_name]
    store = {}
    def fhook(m, i, o):
        store["a"] = o
        o.register_hook(lambda g: store.__setitem__("g", g))
    h = layer.register_forward_hook(fhook)
    try:
        x = x.clone().requires_grad_(True)
        with torch.enable_grad():
            logits = model(x)
            model.zero_grad(set_to_none=True)
            logits.gather(1, target[:, None]).sum().backward()
    finally:
        h.remove()
    A, G = store["a"].detach().float(), store["g"].detach().float()
    cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[:, 0]
    mn = cam.flatten(1).min(1).values.view(-1, 1, 1); mx = cam.flatten(1).max(1).values.view(-1, 1, 1)
    return ((cam - mn) / (mx - mn).clamp_min(1e-8)).cpu()

def cam_overlap(a, b, topk):
    a, b = a.flatten(), b.flatten()
    r = float(np.corrcoef(a.numpy(), b.numpy())[0, 1]) if a.std() > 0 and b.std() > 0 else float("nan")
    k = max(1, int(topk * a.numel()))
    ma = torch.zeros_like(a, dtype=torch.bool); ma[a.topk(k).indices] = True
    mb = torch.zeros_like(b, dtype=torch.bool); mb[b.topk(k).indices] = True
    return r, float((ma & mb).sum() / (ma | mb).sum())

def get_xai(force=False):
    def _compute():
        from sklearn.model_selection import train_test_split
        test_ds = DATA["test_ds"]
        n = min(cfg.xai_n_images, len(test_ds))
        pos = np.arange(len(test_ds)) if n >= len(test_ds) else np.sort(train_test_split(
            np.arange(len(test_ds)), train_size=n, stratify=test_ds.labels, random_state=cfg.seed)[0])
        students = {student_run_name(v): (KD_LABELS[v], get_student(v)["model"])
                    for v in ["nokd", "logit_kd", "dkd", "wsld", "adaptive_kd"] if v in cfg.core_variants}
        for meth in [m for m in ("logit_kd", "adaptive_kd") if m in cfg.recovery_methods]:
            students[pgp_name(SEL_LEVEL, meth, cfg.seed)] = (f"Pruned {int(round(SEL_LEVEL*100))}% · {RECOVERY_LABELS[meth]} "
                                                             "(FP32 proxy for INT8)", get_pgp(SEL_LEVEL, meth, cfg.seed)["model"])
        t_layer = TEACHER_SPECS[cfg.kd_teacher]["cam_layer"]
        rows, examples = [], {}
        for s in range(0, len(pos), 16):
            idx = pos[s:s + 16]
            x = torch.stack([test_ds[int(i)][0] for i in idx]).to(DEVICE)
            y = torch.tensor([test_ds[int(i)][1] for i in idx])
            with torch.no_grad(): t_pred = teacher(x).argmax(1)
            t_cam = gradcam(teacher, t_layer, x, t_pred)
            s_cams = {nm: gradcam(m, STUDENT_CAM_LAYER, x, t_pred) for nm, (_, m) in students.items()}
            for j, i in enumerate(idx):
                for nm in students:
                    r, iou = cam_overlap(t_cam[j], s_cams[nm][j], cfg.xai_topk)
                    rows.append({"test_pos": int(i), "label": int(y[j]), "teacher_pred": int(t_pred[j]),
                                 "model": nm, "pearson_r": r, "iou_topk": iou})
                if len(examples) < 6:
                    examples[int(i)] = {"img": denorm(x[j]), "teacher": t_cam[j].numpy(),
                                        **{nm: s_cams[nm][j].numpy() for nm in students}}
        df = pd.DataFrame(rows); save_df(df, "xai_per_image", XAI_DIR)
        summ = df.groupby("model").agg(pearson_r=("pearson_r", "mean"), pearson_r_std=("pearson_r", "std"),
                                       iou_topk=("iou_topk", "mean"), iou_topk_std=("iou_topk", "std")).reset_index()
        summ["label"] = summ["model"].map({k: v[0] for k, v in students.items()})
        summ = summ.set_index("model").loc[list(students)].reset_index()
        save_df(summ, "table_xai_gradcam_overlap", TABLES_DIR)
        np.savez_compressed(XAI_DIR / "xai_examples.npz", **{f"{i}__{k}": v for i, d in examples.items() for k, v in d.items()})
        # figure
        names = ["teacher"] + list(students)
        fig, axes = plt.subplots(len(examples), len(names) + 1, figsize=(2.1 * (len(names) + 1), 2.1 * len(examples)))
        axes = np.atleast_2d(axes)
        for r_, (i, d) in enumerate(examples.items()):
            axes[r_, 0].imshow(d["img"]); axes[r_, 0].set_title(str(CLASSES[test_ds[i][1]])[:14], fontsize=7)
            for c_, nm in enumerate(names, 1):
                axes[r_, c_].imshow(d["img"]); axes[r_, c_].imshow(d[nm], cmap="jet", alpha=0.45)
                if r_ == 0:
                    axes[r_, c_].set_title("Teacher" if nm == "teacher" else students[nm][0][:28], fontsize=7)
        for a in axes.ravel(): a.axis("off")
        plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_gradcam_teacher_vs_students.png", dpi=250); plt.show()
        return {"summary": summ}
    return ensure_stage("v2_xai_gradcam", _compute,
                        lambda: {"summary": try_load_df("table_xai_gradcam_overlap", TABLES_DIR)}, force=force)

def denorm(t):
    m = torch.tensor(MEAN).view(3, 1, 1); s = torch.tensor(STD).view(3, 1, 1)
    return (t.detach().cpu() * s + m).clamp(0, 1).permute(1, 2, 0).numpy()

XAI = get_xai()
display(XAI["summary"].round(4))
checkpoint_time("S12 XAI done")

---
# S13 · Statistical rigor — multi-seed paired tests  `[stage: v2_stats]`
Each comparison is **proposed vs one baseline**, inside four families: RQ1 dense; RQ2 at each pruning level; INT8 at the selected level; and the compression cost (dense FP32 → pruned INT8). Every comparison reports:
* **Per seed:** McNemar on the same test images (exact if fewer than 25 discordant pairs, otherwise χ² with continuity correction), plus Δacc.
* **Pooled over seeds:** a paired bootstrap that resamples test images (`cfg.bootstrap_n`) on the per-image Δ averaged across seeds. It gives the 95 % CI of the seed-averaged Δacc and a two-sided bootstrap p-value.
* **Holm correction** within each family, and **wins**, e.g. 3/3 seeds.

Claim a difference only when the Holm-corrected p is below .05 *and* the direction is consistent across seeds.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests

def _correct(name):
    y, p = load_preds(name); return y, p.argmax(1) == y

def compare(prop_names, base_names, n_boot=None, rng_seed=0):
    """prop_names / base_names: {seed: preds-name}. Paired over the SAME test images and seeds.
    Returns per-seed McNemar + pooled bootstrap of the seed-averaged Δacc."""
    seeds = [s for s in prop_names if s in base_names
             and (PREDS_DIR / f"{prop_names[s]}.npz").exists() and (PREDS_DIR / f"{base_names[s]}.npz").exists()]
    if not seeds: return None
    A = np.stack([_correct(prop_names[s])[1] for s in seeds]).astype(float)   # seeds × images
    B = np.stack([_correct(base_names[s])[1] for s in seeds]).astype(float)
    per_seed_p, per_seed_d, wins = [], [], 0
    for a, b in zip(A.astype(bool), B.astype(bool)):
        n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
        per_seed_p.append(float(mcnemar([[0, n01], [n10, 0]], exact=(n01 + n10) < 25).pvalue) if n01 + n10 else 1.0)
        per_seed_d.append(100 * (a.mean() - b.mean())); wins += int(a.mean() > b.mean())
    D = (A - B).mean(0)                                   # per-image Δ averaged over seeds
    rng = np.random.default_rng(rng_seed); N = len(D)
    boots = np.array([D[rng.integers(0, N, N)].mean() for _ in range(n_boot or cfg.bootstrap_n)]) * 100
    p_boot = float(min(1.0, 2 * min((boots <= 0).mean(), (boots >= 0).mean())))
    return {"Seeds": len(seeds), "Δacc mean (pp)": float(np.mean(per_seed_d)),
            "Δacc std (pp)": float(np.std(per_seed_d, ddof=1)) if len(seeds) > 1 else np.nan,
            "Δ 95% CI (pp)": f"[{np.percentile(boots, 2.5):+.2f}, {np.percentile(boots, 97.5):+.2f}]",
            "p (pooled bootstrap)": p_boot, "Wins": f"{wins}/{len(seeds)}",
            "McNemar p per seed": ", ".join(f"{p:.3g}" for p in per_seed_p)}

def _family(rows, family):
    df = pd.DataFrame(rows)
    if len(df):
        df.insert(0, "Family", family)
        df["p (Holm)"] = multipletests(df["p (pooled bootstrap)"], method="holm")[1]
        df["Significant (Holm < .05)"] = df["p (Holm)"] < 0.05
    return df

def get_stats(force=False):
    def _compute():
        fams = []
        # RQ1 — dense students
        prop = {s: student_run_name("adaptive_kd", s) for s in SEEDS}
        rows = []
        for v in [v for v in RQ1_ORDER if v != "adaptive_kd"]:
            r = compare(prop, {s: student_run_name(v, s) for s in SEEDS})
            if r: rows.append({"Proposed": KD_LABELS["adaptive_kd"], "Baseline": KD_LABELS[v], **r})
        fams.append(_family(rows, "RQ1 dense FP32"))
        # RQ2 — each pruning level
        for lv in cfg.prune_levels:
            prop = {s: pgp_name(lv, "adaptive_kd", s) for s in SEEDS}
            rows = []
            for m in [m for m in cfg.recovery_methods if m != "adaptive_kd"]:
                r = compare(prop, {s: pgp_name(lv, m, s) for s in SEEDS})
                if r: rows.append({"Proposed": RECOVERY_LABELS["adaptive_kd"], "Baseline": RECOVERY_LABELS[m], **r})
            fams.append(_family(rows, f"RQ2 pruned {int(round(lv*100))}% FP32"))
        # INT8 at the selected level
        prop = {s: f"int8_{pgp_name(SEL_LEVEL, 'adaptive_kd', s)}_qat" for s in SEEDS}
        rows = []
        for m in [m for m in QAT_METHODS if m != "adaptive_kd"] + (["none"] if "none" in cfg.recovery_methods else []):
            rec = "ptq" if m == "none" else "qat"
            r = compare(prop, {s: f"int8_{pgp_name(SEL_LEVEL, m, s)}_{rec}" for s in SEEDS})
            if r: rows.append({"Proposed": "Adaptive KD · INT8-QAT", "Baseline": f"{RECOVERY_LABELS[m]} · INT8-{rec.upper()}", **r})
        fams.append(_family(rows, f"INT8 pruned {int(round(SEL_LEVEL*100))}%"))
        # compression cost of the proposed pipeline (dense FP32 → pruned INT8)
        r = compare(prop, {s: student_run_name("adaptive_kd", s) for s in SEEDS})
        if r: fams.append(_family([{"Proposed": "Adaptive KD · pruned INT8-QAT", "Baseline": "Adaptive KD · dense FP32", **r}],
                                  "Compression cost"))
        df = pd.concat([f for f in fams if len(f)], ignore_index=True)
        save_df(df, "table_statistics", TABLES_DIR)
        return {"df": df}
    return ensure_stage("v2_stats", _compute, lambda: {"df": try_load_df("table_statistics", TABLES_DIR)}, force=force)

STATS = get_stats()
pd.set_option("display.max_colwidth", 60)
display(STATS["df"].round(4))

---
# S14 · Results — experimental matrix, hard-subset analysis, figures, LaTeX
Everything here is **read from Drive** (stored predictions, benchmark and stage JSONs). Accuracy is mean ± std over seeds; size and latency come from the main seed, since they depend only on the architecture.

| # | Model | Precision | KD |
|---|---|---|---|
| 1 | Teacher | FP32 | — |
| 2 | FasterCellNet | FP32 | No KD |
| 3 | FasterCellNet | FP32 | Logit KD |
| 4 | FasterCellNet | FP32 | best modern baseline (by val) |
| 5 | FasterCellNet | FP32 | Difficulty-aware KD |
| 6 | Pruned FasterCellNet | FP32 | Difficulty-aware KD |
| 7 | Pruned | INT8-PTQ | No recovery |
| 8–9 | Pruned | INT8-QAT | Logit KD / best baseline |
| **10** | **Pruned** | **INT8-QAT** | **Difficulty-aware KD (proposed)** |

**Hard-subset analysis.** This is accuracy on the third of test images the *teacher* is least certain about (entropy + 1 − confidence). That is where a difficulty-aware method should differ from the others, if it differs anywhere.

In [ ]:
def _pm(name):
    return json.loads((PREDS_DIR / f"{name}.json").read_text())

def _ms(names):
    """names: {seed: preds-name} → mean/std of acc, macro-F1, AUC, ECE over the seeds that exist."""
    ms = [_pm(n) for n in names.values() if (PREDS_DIR / f"{n}.json").exists()]
    out = {"Seeds": len(ms)}
    for k, col in (("acc", "Acc"), ("f1_macro", "Macro-F1"), ("auc_ovr", "AUC"), ("ece", "ECE")):
        v = np.array([m[k] for m in ms]); out[col] = v.mean(); out[col + " std"] = v.std(ddof=1) if len(v) > 1 else np.nan
    return out

def best_dense_baseline():
    cands = [v for v in cfg.core_variants if v in ("dkd", "dist", "ctkd", "wsld", "ls_kd")]
    if not cands: return None
    val = {v: np.mean([m["acc"] for m in _seed_metrics(v, split="val").values()]) for v in cands}
    return max(val, key=val.get)

PROPOSED = f"int8_{pgp_name(SEL_LEVEL, 'adaptive_kd', cfg.seed)}_qat"
BEST_DENSE = best_dense_baseline()
P_ = f"Pruned FasterCellNet ({int(round(SEL_LEVEL*100))}%)"
_by = lambda f: {s: f(s) for s in SEEDS}

spec = [  # (#, model, precision, KD, names-by-seed, representative name for size/latency, fp32 model or None)
    (1, f"Teacher ({cfg.teacher_arch})", "FP32", "—", {cfg.seed: f"teacher_{cfg.kd_teacher}"}, f"teacher_{cfg.kd_teacher}", teacher),
    (2, "FasterCellNet", "FP32", "No KD", _by(lambda s: student_run_name("nokd", s)), student_run_name("adaptive_kd"), get_student("nokd")["model"]),
    (3, "FasterCellNet", "FP32", "Logit KD", _by(lambda s: student_run_name("logit_kd", s)), student_run_name("adaptive_kd"), get_parent_student()),
]
if BEST_DENSE:
    spec.append((4, "FasterCellNet", "FP32", KD_LABELS[BEST_DENSE] + " (best baseline, val)",
                 _by(lambda s: student_run_name(BEST_DENSE, s)), student_run_name("adaptive_kd"), get_parent_student()))
spec += [
    (5, "FasterCellNet", "FP32", "Difficulty-aware KD", _by(lambda s: student_run_name("adaptive_kd", s)), student_run_name("adaptive_kd"), get_parent_student()),
    (6, P_, "FP32", "Difficulty-aware KD", _by(lambda s: pgp_name(SEL_LEVEL, "adaptive_kd", s)),
     pgp_name(SEL_LEVEL, "adaptive_kd", cfg.seed), get_pgp(SEL_LEVEL, "adaptive_kd", cfg.seed)["model"]),
]
if "none" in cfg.recovery_methods:
    spec.append((7, P_, "INT8-PTQ", "No recovery", _by(lambda s: f"int8_{pgp_name(SEL_LEVEL, 'none', s)}_ptq"),
                 f"int8_{pgp_name(SEL_LEVEL, 'none', cfg.seed)}_ptq", None))
for i, m in enumerate([m for m in QAT_METHODS if m != "adaptive_kd"]):
    spec.append((8 + i, P_, "INT8-QAT", RECOVERY_LABELS[m], _by(lambda s, m=m: f"int8_{pgp_name(SEL_LEVEL, m, s)}_qat"),
                 f"int8_{pgp_name(SEL_LEVEL, m, cfg.seed)}_qat", None))
spec.append((10, P_ + " — proposed", "INT8-QAT", "Difficulty-aware KD", _by(lambda s: f"int8_{pgp_name(SEL_LEVEL, 'adaptive_kd', s)}_qat"),
             PROPOSED, None))

rows = []
for num, model, prec, kd, names, rep, fp32 in spec:
    r = {"#": num, "Model": model, "Precision": prec, "KD": kd, **_ms(names)}
    if fp32 is not None:
        r["Params (M)"], r["Size (MB)"] = count_params(fp32) / 1e6, disk_size_mb(fp32)
    else:
        info = load_json(f"{rep}_info"); r["Params (M)"], r["Size (MB)"] = info["params_fp32"] / 1e6, info["size_mb"]
    b = BENCH.get(rep, {})
    r["Latency mean (ms)"], r["Latency p95 (ms)"] = b.get("latency_mean_ms"), b.get("latency_p95_ms")
    rows.append(r)
MATRIX = pd.DataFrame(rows)
t_size, t_acc = MATRIX.loc[0, "Size (MB)"], MATRIX.loc[0, "Acc"]
MATRIX["Compression vs teacher (x)"] = t_size / MATRIX["Size (MB)"]
MATRIX["Δacc vs teacher (pp)"] = 100 * (MATRIX["Acc"] - t_acc)
MATRIX["Acc (%)"] = [f"{100*m:.2f}" + ("" if np.isnan(sd) else f" ± {100*sd:.2f}") for m, sd in zip(MATRIX["Acc"], MATRIX["Acc std"])]
MATRIX["Macro-F1 (%)"] = [f"{100*m:.2f}" + ("" if np.isnan(sd) else f" ± {100*sd:.2f}") for m, sd in zip(MATRIX["Macro-F1"], MATRIX["Macro-F1 std"])]

dep_rows = []
for key, ex in EXPORTS.items():
    o = ex["onnx"]
    for tag, size, met in (("onnx_fp32", o["fp32_size_mb"], o["onnx_fp32_test"]), ("onnx_int8", o["int8_size_mb"], o["onnx_int8_test"])):
        dep_rows.append({"Model": f"{key} · {tag}", "Size (MB)": size, "Acc": met["acc"], "Macro-F1": met["f1_macro"],
                         "ECE": met["ece"], "Latency mean (ms)": BENCH.get(f"{tag}_{key}", {}).get("latency_mean_ms")})
    if ex.get("tflite"):
        for tag in ("tflite_fp32", "tflite_int8"):
            t = ex["tflite"][tag]
            dep_rows.append({"Model": f"{key} · {tag}", "Size (MB)": t["size_mb"], "Acc": t["test"]["acc"],
                             "Macro-F1": t["test"]["f1_macro"], "ECE": t["test"]["ece"],
                             "Latency mean (ms)": BENCH.get(f"{tag}_{key}", {}).get("latency_mean_ms")})
DEPLOY_DF = pd.DataFrame(dep_rows)

# ---- hard-subset analysis: accuracy on the test images the TEACHER finds hardest -------
y_t, p_t = load_preds(f"teacher_{cfg.kd_teacher}")
_H = -(p_t * np.log(np.clip(p_t, 1e-12, 1))).sum(1) / np.log(NUM_CLASSES)
_Dt = (_H + 1 - p_t.max(1)) / 2
HARD = _Dt.argsort().argsort() >= (2 / 3) * (len(_Dt) - 1)
def _hard_acc(names):
    v = [ (load_preds(n)[1].argmax(1) == y_t)[HARD].mean() for n in names.values() if (PREDS_DIR / f"{n}.npz").exists()]
    return np.mean(v), (np.std(v, ddof=1) if len(v) > 1 else np.nan), len(v)
hard_rows = []
for v in RQ1_ORDER:
    m, sd, n = _hard_acc(_by(lambda s: student_run_name(v, s)))
    hard_rows.append({"Setting": "Dense FP32", "Method": KD_LABELS[v], "Seeds": n, "Hard-⅓ acc": m, "std": sd})
for lv in cfg.prune_levels:
    for meth in cfg.recovery_methods:
        m, sd, n = _hard_acc(_by(lambda s: pgp_name(lv, meth, s)))
        hard_rows.append({"Setting": f"Pruned {int(round(lv*100))}%", "Method": RECOVERY_LABELS[meth], "Seeds": n, "Hard-⅓ acc": m, "std": sd})
HARD_DF = pd.DataFrame(hard_rows)
print(f"Hard third = {HARD.sum()} test images; holds {(p_t.argmax(1) != y_t)[HARD].sum()} of the teacher's "
      f"{(p_t.argmax(1) != y_t).sum()} errors")
save_df(HARD_DF, "table_hard_subset", TABLES_DIR); display(HARD_DF.round(4))

def save_latex(df, name, caption, cols=None, fmt="%.4f"):
    d = (df[cols] if cols else df).copy()
    body = d.to_latex(index=False, float_format=lambda v: fmt % v, escape=True, na_rep="--")
    tex = ("\\begin{table*}[t]\n\\centering\n\\caption{" + caption + "}\n\\label{tab:" + name + "_" + cfg.dataset + "}\n"
           "\\resizebox{\\textwidth}{!}{%\n" + body + "}\n\\end{table*}\n")
    (TABLES_DIR / f"{name}.tex").write_text(tex)

DS_NAME = {"bloodmnist": "BloodMNIST", "raabin": "Raabin-WBC"}.get(cfg.dataset, cfg.dataset)
nseed = f"mean ± std over {len(SEEDS)} seeds"
MATRIX_COLS = ["#", "Model", "Precision", "KD", "Params (M)", "Size (MB)", "Acc (%)", "Macro-F1 (%)", "ECE",
               "Latency mean (ms)", "Compression vs teacher (x)", "Δacc vs teacher (pp)"]
save_df(MATRIX, "table_experimental_matrix", TABLES_DIR); save_df(DEPLOY_DF, "table_deployed_runtimes", TABLES_DIR)
save_latex(MATRIX, "table_experimental_matrix", f"Experimental matrix on {DS_NAME} (test set, {nseed}). Latency: batch-1 host CPU, main seed.", MATRIX_COLS)
save_latex(fmt_pm(RQ1_DF), "table_rq1_knowledge_transfer", f"RQ1 — knowledge transfer on {DS_NAME} ({nseed}; ablations: 1 seed).")
save_latex(RQ2_PM, "table_rq2_pruning_recovery", f"RQ2 — KD-guided progressive pruning on {DS_NAME} ({nseed}).")
save_latex(ROB_SUMMARY, "table_rq3_robustness", f"RQ3 — robustness on {DS_NAME}: 7 corruptions x 3 severities"
           + (" + external TestB" if external_loader is not None else "") + f" ({nseed}).")
save_latex(XAI["summary"].drop(columns=["model"]), "table_xai_gradcam_overlap", f"Grad-CAM overlap with the teacher ({DS_NAME}, main seed).")
save_latex(STATS["df"], "table_statistics", f"Paired comparisons vs. the proposed method on {DS_NAME}: per-seed McNemar, "
           "pooled bootstrap 95\\% CI of the seed-averaged accuracy difference, Holm-corrected within each family.")
save_latex(HARD_DF, "table_hard_subset", f"Accuracy on the teacher-hardest third of the {DS_NAME} test set.")
display(MATRIX[MATRIX_COLS].round(4)); display(DEPLOY_DF.round(4))

# ---- Pareto: accuracy vs size and vs latency --------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
for ax, xcol, xl in ((axes[0], "Size (MB)", "Model size (MB, log)"), (axes[1], "Latency mean (ms)", "CPU latency, batch 1 (ms, log)")):
    d = MATRIX.dropna(subset=[xcol])
    ax.errorbar(d[xcol], 100 * d["Acc"], yerr=100 * d["Acc std"].fillna(0), fmt="none", ecolor="gray", capsize=2, zorder=2)
    ax.scatter(d[xcol], 100 * d["Acc"], s=[140 if n == 10 else 60 for n in d["#"]],
               c=["#C44E52" if n == 10 else "#4C72B0" for n in d["#"]], zorder=3)
    for _, r in d.iterrows():
        ax.annotate(str(int(r["#"])), (r[xcol], 100 * r["Acc"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
    pts = d.sort_values(xcol)[[xcol, "Acc"]].values; front, best = [], -1
    for x_, a_ in pts:
        if a_ > best: front.append((x_, 100 * a_)); best = a_
    ax.plot(*zip(*front), "--", c="gray", lw=1, label="Pareto front")
    ax.set_xscale("log"); ax.set_xlabel(xl); ax.set_ylabel("Test accuracy (%)"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.suptitle("Accuracy–efficiency trade-off (numbers = matrix rows; red = proposed)")
plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_pareto_matrix.png", dpi=300); plt.show()

# ---- confusion + per-class + reliability for the proposed model ---------------
y_t, p_t = load_preds(f"teacher_{cfg.kd_teacher}"); y_p, p_p = load_preds(PROPOSED)
plot_confusion(y_p, p_p, "Proposed: pruned INT8 FasterCellNet (adaptive KD)", "fig_cm_proposed.png")
rep = classification_report(y_p, p_p.argmax(1), labels=list(range(NUM_CLASSES)), target_names=[str(c) for c in CLASSES],
                            output_dict=True, zero_division=0)
PER_CLASS = pd.DataFrame(rep).T.round(4)
save_df(PER_CLASS.reset_index().rename(columns={"index": "class"}), "table_per_class_proposed", TABLES_DIR)
display(PER_CLASS)

fig, ax = plt.subplots(figsize=(5.2, 5))
for nm, lab in ((f"teacher_{cfg.kd_teacher}", "Teacher"), (student_run_name("adaptive_kd"), "Dense adaptive KD"), (PROPOSED, "Proposed INT8")):
    y_, p_ = load_preds(nm); conf, corr = p_.max(1), (p_.argmax(1) == y_)
    bins = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 0: xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax.plot(xs, ys, "o-", label=f"{lab} (ECE {expected_calibration_error(p_, y_):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("confidence"); ax.set_ylabel("accuracy")
ax.set_title("Reliability diagram"); ax.legend(fontsize=8); plt.tight_layout()
plt.savefig(FIGS_DIR / "fig_reliability.png", dpi=300); plt.show()

# ---- RQ1 validation curves -----------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4.2))
for v in RQ1_ORDER:
    h = try_load_df(f"{student_run_name(v)}_history")
    if h is not None and len(h): ax.plot(h["epoch"], 100 * h["f1_macro"], label=KD_LABELS[v])
ax.set_xlabel("epoch"); ax.set_ylabel("val macro-F1 (%)"); ax.set_title(f"RQ1 — validation curves (main seed, {DS_NAME})"); ax.legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_rq1_val_curves.png", dpi=300); plt.show()

# ---- difficulty analysis (adaptive run) ------------------------------------------
dfiles = sorted((RUNS_DIR / student_run_name("adaptive_kd")).glob("difficulty_ep*.npy"))
if len(dfiles) >= 2:
    tr_labels = DATA["train_ds"].labels
    def buckets(D):
        rk = np.empty(len(D)); rk[np.argsort(D)] = np.arange(len(D)) / max(len(D) - 1, 1)
        return np.minimum((rk * 3).astype(int), 2)
    first_kl = dfiles[min(cfg.diff_warmup_epochs, len(dfiles) - 1)]
    b0, b1 = buckets(np.load(first_kl)), buckets(np.load(dfiles[-1]))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    comp = pd.crosstab(pd.Series([str(CLASSES[i])[:12] for i in tr_labels], name="class"),
                       pd.Series(np.array(["easy", "medium", "hard"])[b1], name="bucket"), normalize="index")
    comp = comp.reindex(columns=["easy", "medium", "hard"], fill_value=0)
    comp.plot(kind="bar", stacked=True, ax=axes[0], color=["#55A868", "#DD8452", "#C44E52"])
    axes[0].set_ylabel("fraction of training images"); axes[0].set_title("Difficulty buckets per class (last epoch)")
    trans = pd.crosstab(b0, b1, normalize="index").reindex(index=range(3), columns=range(3), fill_value=0)
    im = axes[1].imshow(trans.values, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3): axes[1].text(j, i, f"{trans.values[i, j]:.2f}", ha="center", va="center")
    axes[1].set_xticks(range(3)); axes[1].set_xticklabels(["easy", "medium", "hard"])
    axes[1].set_yticks(range(3)); axes[1].set_yticklabels(["easy", "medium", "hard"])
    axes[1].set_xlabel(f"bucket at {dfiles[-1].stem}"); axes[1].set_ylabel(f"bucket at {first_kl.stem}")
    axes[1].set_title("Bucket migration during training")
    plt.tight_layout(); plt.savefig(FIGS_DIR / "fig_difficulty_analysis.png", dpi=300); plt.show()
checkpoint_time("S14 results done")

---
# S15 · RQ4 — On-device benchmark harness (HADAC smartphones)
A notebook can't measure phone energy or temperatures, so this cell writes a self-contained package to `device_benchmark/` on Drive:

* `models/` — the exported TFLite (FP32, INT8) and ONNX files for the dense and pruned models
* `benchmark_spec.json` — the protocol: model list, warm-up, timed runs, sustained-inference duration, thread counts, delegates, sampling rates
* `results_template.csv` — one row per (device, model, delegate, threads), with the columns the ingest cell expects
* `README_device_protocol.md` — `adb` commands for the TFLite `benchmark_model` binary and for reading current and thermal zones, plus the energy formula

Fill the CSV from your HADAC instrumentation, save it as `device_benchmark/results/<device>.csv`, then run **S16**.

In [ ]:
DEV_MODELS_DIR = DEVICE_DIR / "models"; DEV_MODELS_DIR.mkdir(exist_ok=True)
(DEVICE_DIR / "results").mkdir(exist_ok=True)

dev_models = []
for key, ex in EXPORTS.items():
    cands = [("onnx_fp32", ex["onnx"]["fp32_path"]), ("onnx_int8", ex["onnx"]["int8_path"])]
    if ex.get("tflite"):
        cands += [("tflite_fp32", ex["tflite"]["tflite_fp32"]["path"]), ("tflite_int8", ex["tflite"]["tflite_int8"]["path"])]
    for tag, p in cands:
        dst = DEV_MODELS_DIR / f"fastercellnet_{key}_{tag.split('_', 1)[1]}.{tag.split('_')[0]}"
        shutil.copy(p, dst)          # always refresh (model may have changed since an earlier run)
        acc = (ex["onnx"][f"{tag}_test"]["acc"] if tag.startswith("onnx") else ex["tflite"][tag]["test"]["acc"])
        dev_models.append({"model_id": f"{key}_{tag}", "file": dst.name, "runtime": tag.split("_")[0],
                           "precision": tag.split("_")[1].upper(), "size_mb": round(os.path.getsize(dst) / 1e6, 3),
                           "test_acc_measured": round(acc, 4)})

spec = {
    "input": {"shape_nhwc": [1, cfg.img_size, cfg.img_size, 3], "normalization": {"mean": MEAN, "std": STD}},
    "models": dev_models,
    "protocol": {"warmup_runs": 50, "timed_runs": 500, "sustained_minutes": 10, "cooldown_minutes": 5,
                 "threads": [1, 4], "delegates": ["cpu", "xnnpack", "gpu", "nnapi"],
                 "sampling_hz": {"current": 10, "thermal": 1}, "screen": "on, min brightness, airplane mode",
                 "battery": "40–80 %, not charging", "repeats_per_config": 3},
    "metrics": ["latency_mean_ms", "latency_p50_ms", "latency_p95_ms", "latency_std_ms", "fps", "peak_memory_mb",
                "init_time_ms", "avg_current_ma", "avg_voltage_v", "idle_power_mw", "energy_per_inference_mj",
                "soc_temp_start_c", "soc_temp_end_c", "skin_temp_end_c", "battery_temp_end_c",
                "latency_first_min_ms", "latency_last_min_ms", "throttling_ratio"],
}
atomic_write_text(DEVICE_DIR / "benchmark_spec.json", json.dumps(spec, indent=2))

cols = ["device", "soc", "android_version", "model_id", "delegate", "threads"] + spec["metrics"]
tmpl = pd.DataFrame([{c: "" for c in cols} | {"model_id": m["model_id"]} for m in dev_models if m["runtime"] == "tflite"] or
                    [{c: "" for c in cols}])
tmpl.to_csv(DEVICE_DIR / "results_template.csv", index=False)

readme = f"""# FasterCellNet-RKD — on-device protocol

## Latency / memory (TFLite benchmark_model)
```bash
adb push models/*.tflite /data/local/tmp/
adb push android_aarch64_benchmark_model /data/local/tmp/benchmark_model && adb shell chmod +x /data/local/tmp/benchmark_model
# CPU / XNNPACK, 1 and 4 threads
adb shell /data/local/tmp/benchmark_model --graph=/data/local/tmp/fastercellnet_pruned_int8.tflite \\
    --num_threads=4 --use_xnnpack=true --warmup_runs=50 --num_runs=500 --report_peak_memory_footprint=true
# GPU / NNAPI delegates
adb shell /data/local/tmp/benchmark_model --graph=... --use_gpu=true   --num_runs=500
adb shell /data/local/tmp/benchmark_model --graph=... --use_nnapi=true --num_runs=500
```
`init_time_ms` = "Initialized session in" line · `latency_*` = "Inference timings in us" (÷1000) · `peak_memory_mb` = peak memory footprint.

## Sustained run (throttling)
Run `--num_runs` large enough for {spec['protocol']['sustained_minutes']} min (or `--duration`/looped invocations); record latency over the first and last minute →
`throttling_ratio = latency_last_min_ms / latency_first_min_ms`.

## Power / energy
```bash
adb shell cat /sys/class/power_supply/battery/current_now   # µA (sign convention varies by vendor)
adb shell cat /sys/class/power_supply/battery/voltage_now   # µV
adb shell dumpsys battery                                  # battery temperature (tenths of °C)
adb shell 'for z in /sys/class/thermal/thermal_zone*; do echo $(cat $z/type) $(cat $z/temp); done'
```
Sample current at {spec['protocol']['sampling_hz']['current']} Hz during the timed loop and at idle (same screen state).
`energy_per_inference_mj = (P_active_mW − P_idle_mW) × latency_mean_ms / 1000`, with `P = |I| × V`.
Skin temperature: external IR/thermocouple on the SoC side of the back cover.

## Deliver
Fill `results_template.csv` per device → save as `results/<device>.csv` on Drive → run notebook cell S16.
"""
(DEVICE_DIR / "README_device_protocol.md").write_text(readme)
display(pd.DataFrame(dev_models))
print("On-device package written to", DEVICE_DIR)

---
# S16 · RQ4 — Ingest smartphone results and build the edge Pareto front
Reads every `device_benchmark/results/*.csv`, joins each row with the model's measured test accuracy, and plots accuracy vs latency and accuracy vs energy per device, with temperature rise and throttling as a table. Until results exist, this cell just says so. It is safe to re-run any time you add a device.

In [ ]:
res_files = sorted((DEVICE_DIR / "results").glob("*.csv"))
if not res_files:
    print("No on-device results yet — fill device_benchmark/results_template.csv and save it to device_benchmark/results/.")
else:
    DEV = pd.concat([pd.read_csv(f) for f in res_files], ignore_index=True)
    acc_map = {m["model_id"]: m["test_acc_measured"] for m in json.loads((DEVICE_DIR / "benchmark_spec.json").read_text())["models"]}
    DEV["test_acc"] = DEV["model_id"].map(acc_map)
    for c in ["latency_mean_ms", "energy_per_inference_mj", "soc_temp_start_c", "soc_temp_end_c", "throttling_ratio"]:
        DEV[c] = pd.to_numeric(DEV.get(c), errors="coerce")
    DEV["soc_temp_rise_c"] = DEV["soc_temp_end_c"] - DEV["soc_temp_start_c"]
    save_df(DEV, "table_rq4_on_device", TABLES_DIR)
    display(DEV)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for (dev, dl), g in DEV.groupby(["device", "delegate"]):
        axes[0].scatter(g["latency_mean_ms"], 100 * g["test_acc"], label=f"{dev} · {dl}")
        if g["energy_per_inference_mj"].notna().any():
            axes[1].scatter(g["energy_per_inference_mj"], 100 * g["test_acc"], label=f"{dev} · {dl}")
        for _, r in g.iterrows():
            axes[0].annotate(r["model_id"], (r["latency_mean_ms"], 100 * r["test_acc"]), fontsize=6)
    axes[0].set_xlabel("on-device latency (ms)"); axes[1].set_xlabel("energy / inference (mJ)")
    for a in axes: a.set_ylabel("test accuracy (%)"); a.grid(alpha=.3); a.legend(fontsize=7)
    fig.suptitle("RQ4 — accuracy vs latency / energy on smartphones"); plt.tight_layout()
    plt.savefig(FIGS_DIR / "fig_rq4_device_pareto.png", dpi=300); plt.show()
    display(DEV.groupby(["device", "model_id", "delegate"])[["latency_mean_ms", "energy_per_inference_mj",
                                                             "soc_temp_rise_c", "throttling_ratio"]].mean().round(3))

---
# S16b · Cross-dataset summary (BloodMNIST × Raabin-WBC)
Reads the finished tables of **both** datasets from Drive. Neither dataset needs to be loaded in this session, and if only one is finished, the cell shows what is available. The output goes to `/MyDrive/FasterCellNet_RKD/cross_dataset/`: combined RQ1 / RQ2 / RQ3 tables, a combined significance summary, and LaTeX versions of each.

In [ ]:
CROSS_DIR = DRIVE_ROOT / "cross_dataset"; CROSS_DIR.mkdir(exist_ok=True)
DS_ROOTS = {"BloodMNIST": DRIVE_ROOT, "Raabin-WBC": DRIVE_ROOT / "raabin"}
if MODE == "smoke":
    DS_ROOTS = {"synthetic": DRIVE_ROOT / "synthetic", "raabin": DRIVE_ROOT / "raabin"}

def _read(root, name):
    p = Path(root) / "tables" / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else None

def _wide(name, key_cols, val_col, rename=None):
    parts = []
    for ds, root in DS_ROOTS.items():
        d = _read(root, name)
        if d is None or val_col not in d.columns: continue
        d = d[key_cols + [val_col]].rename(columns={val_col: ds}); parts.append(d.set_index(key_cols))
    return pd.concat(parts, axis=1).reset_index() if parts else None

available = {ds: (Path(r) / "tables" / "table_rq1_knowledge_transfer_pm.csv").exists() for ds, r in DS_ROOTS.items()}
print("Datasets with finished tables:", available)

CROSS = {
    "rq1_accuracy": _wide("table_rq1_knowledge_transfer_pm", ["Variant"], "acc"),
    "rq1_macro_f1": _wide("table_rq1_knowledge_transfer_pm", ["Variant"], "f1_macro"),
    "rq2_accuracy": _wide("table_rq2_pruning_recovery_pm", ["Pruned", "Recovery"], "Test acc (%)"),
    "rq3_mCA": _wide("table_rq3_robustness_summary", ["group"], "mCA"),
    "hard_subset": _wide("table_hard_subset", ["Setting", "Method"], "Hard-⅓ acc"),
}
sig_parts = []
for ds, root in DS_ROOTS.items():
    d = _read(root, "table_statistics")
    if d is not None:
        d = d[["Family", "Baseline", "Seeds", "Δacc mean (pp)", "Δ 95% CI (pp)", "Wins", "p (Holm)"]].copy()
        d.insert(0, "Dataset", ds); sig_parts.append(d)
CROSS["significance"] = pd.concat(sig_parts, ignore_index=True) if sig_parts else None

for k, df in CROSS.items():
    if df is None: continue
    df.to_csv(CROSS_DIR / f"cross_{k}.csv", index=False)
    body = df.to_latex(index=False, float_format=lambda v: f"{v:.4f}", escape=True, na_rep="--")
    (CROSS_DIR / f"cross_{k}.tex").write_text("\\begin{table*}[t]\n\\centering\n\\caption{Cross-dataset: "
                                              + k.replace("_", " ") + "}\n\\resizebox{\\textwidth}{!}{%\n" + body + "}\n\\end{table*}\n")
    print(f"\n=== {k} ==="); display(df.round(4) if k in ("rq3_mCA", "hard_subset", "significance") else df)

if CROSS["significance"] is not None:
    s = CROSS["significance"]
    s = s.assign(direction=np.sign(s["Δacc mean (pp)"]), sig=s["p (Holm)"] < 0.05)
    print("\nSummary — proposed vs baselines (Holm-corrected, p<.05):")
    display(s.groupby(["Dataset", "Family"]).agg(comparisons=("Baseline", "count"),
                                                 proposed_better_sig=("sig", lambda x: int((x & (s.loc[x.index, "direction"] > 0)).sum())),
                                                 proposed_worse_sig=("sig", lambda x: int((x & (s.loc[x.index, "direction"] < 0)).sum()))))

---
# S17 · Manifest and archive
Lists every figure and table, then zips `figs/`, `tables/`, `export/`, `device_benchmark/`, the stage JSONs and the stored predictions into one archive on Drive. Model weights stay in `ckpt/` and `runs/`, so the archive stays small.

In [ ]:
import zipfile
manifest = pd.DataFrame([{"type": "figure", "file": f.name} for f in sorted(FIGS_DIR.glob("*.png"))] +
                        [{"type": "table", "file": f.name} for f in sorted(TABLES_DIR.glob("*.*"))])
save_df(manifest, "manifest", TABLES_DIR)
display(manifest)

zip_path = DATA_ROOT / f"FasterCellNet_RKD_v2_results_{cfg.dataset}_{MODE}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for d in (FIGS_DIR, TABLES_DIR, DEVICE_DIR, PREDS_DIR, ROBUST_DIR, XAI_DIR):
        for f in d.rglob("*"):
            if f.is_file() and not f.name.endswith(".zip"): z.write(f, f.relative_to(DATA_ROOT))
    for f in list(CKPT_DIR.glob("*.json")) + list(CKPT_DIR.glob("*.csv")) + list(EXPORT_DIR.glob("*.json")):
        z.write(f, f.relative_to(DATA_ROOT))
    for f in (STATUS_PATH, _cfg_marker):
        if f.exists(): z.write(f, f.relative_to(DATA_ROOT))
print(f"Archive: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
show_status()
checkpoint_time("Notebook complete")